In [ ]:
# -*- coding: utf-8 -*-

BS-TF LSTM_FIX.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1RkDgEYW4bnePHrGqb9hFukJ1kDAdXW4W

## CATATAN VERSI DIPERBAIKI (bs_tf_lstm_fix_fixed)

File ini adalah salinan bs_tf_lstm_fix.py yang SUDAH DIPERBAIKI untuk
mengatasi temuan data leakage (scaler fit-before-split) dari audit --
lihat audit/results/audit_findings.md di repo untuk detail lengkap.

Ada 3 lokasi yang diperbaiki (ditandai komentar "[FIX LEAKAGE #N]" di
kodenya), semuanya mengubah urutan `scaler.fit_transform(seluruh_data)`
menjadi `scaler.fit(hanya_data_train)` lalu `scaler.transform(seluruh_data)`:
1. Baseline (univariate regional) -- scaler kini hanya fit dari baris tahun
   <=2024 (periode train+val), bukan seluruh seri termasuk 2025 (lihat juga
   "[FIX SPLIT #1]" di bawah).
2. Pre-Training nasional -- scaler kini hanya fit dari baris tahun <=2023
   (periode train), bukan ikut 2024 (periode validasi).
3. Regional (dipakai bersama tahap Fine-Tuning + Iterasi 1/2/3) -- scaler
   kini hanya fit dari baris tahun <=2024 (periode fine-tuning), bukan
   ikut tahun 2025 (data uji akhir).

Tahap "MODEL FINAL" SENGAJA TIDAK diubah -- tahap itu memang melatih
memakai SELURUH data 2023-2025 tanpa menyisakan data uji sama sekali
(lihat komentar "TANPA menyisakan data uji" di kodenya), jadi tidak ada
split untuk bocor.

Akibat perbaikan ini, angka RMSE/MAE/MAPE pada Baseline/Pre-Training/
Fine-Tuning/Iterasi 1-3 akan SEDIKIT BERBEDA dari hasil run sebelumnya
(lihat audit/results/eval_summary_before_leakage_fix.csv untuk angka
versi lama sebagai pembanding).

## CATATAN TASK GROUP 1 (perbaikan split Baseline)

Selain FIX LEAKAGE di atas, split Baseline (ditandai komentar
"[FIX SPLIT #N]") diubah dari rasio index 80/20 (`TRAIN_RATIO`) menjadi
time-based per tahun kalender, identik polanya dengan Fine-Tuning: train+val
= 2023-2024 (dipecah lagi 85/15 pakai `VAL_INTERNAL_RATIO` untuk validasi
internal), test = 2025. Ini menyamakan periode data uji Baseline dengan
periode data uji varian lain, dan membuat Baseline sekarang divalidasi
memakai `X_val`/`y_val` saat training (sebelumnya divalidasi memakai
`X_test`/`y_test`, yang juga merupakan jalur leakage tersendiri).

LIBRARAY

In [ ]:
# Library untuk manipulasi data
import pandas as pd
import numpy as np

# Library untuk visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# Library untuk preprocessing data
from sklearn.preprocessing import MinMaxScaler

# Library untuk membangun model LSTM (Deep Learning)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Library untuk evaluasi model
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Mengatur random seed agar hasil eksperimen konsisten (reproducible)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Mengatur tampilan visualisasi
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

print("Seluruh library berhasil diimpor.")
print("TensorFlow version:", tf.__version__)

DATASET REGIONAL

In [ ]:
# Path dataset regional Sulawesi Selatan
# Silakan sesuaikan path file dengan lokasi dataset pada Google Drive/Colab
DATASET_PATH = "DATA_PHASE_3_REGIONAL_MODIFIED.csv"

# Membaca dataset
df = pd.read_csv(DATASET_PATH)

# Menampilkan 5 baris pertama data
df.head()
# Menampilkan informasi umum dataset (jumlah baris, tipe data, missing value per kolom)
print("Informasi Dataset:")
df.info()
# Menampilkan statistik deskriptif dari dataset (mean, std, min, max, dsb)
print("Statistik Deskriptif Dataset:")
df.describe(include="all")
# Mengecek jumlah missing value pada setiap kolom
print("Jumlah Missing Value per Kolom:")
print(df.isnull().sum())

print("Distribusi Jumlah Data per Jenis PLT:")
print(df["Jenis"].value_counts())

# Visualisasi distribusi jumlah data per jenis PLT
plt.figure(figsize=(10, 4))
sns.countplot(data=df, x="Jenis", order=df["Jenis"].value_counts().index)
plt.title("Distribusi Jumlah Data per Jenis PLT")
plt.xlabel("Jenis PLT")
plt.ylabel("Jumlah Data")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Melihat distribusi nilai produksi EBT (variabel target) secara keseluruhan
plt.figure(figsize=(10, 4))
sns.histplot(df["Produksi"], bins=30, kde=True, color="seagreen")
plt.title("Distribusi Nilai Produksi EBT (GWh)")
plt.xlabel("Produksi (GWh)")
plt.ylabel("Frekuensi")
plt.tight_layout()
plt.show()

# 1. Mengonversi kolom tanggal menjadi tipe data datetime
df["Tanggal"] = pd.to_datetime(df["Tanggal"])

# 2. Mengurutkan data berdasarkan jenis PLT lalu berdasarkan waktu (tanggal)
df = df.sort_values(by=["Jenis", "Tanggal"]).reset_index(drop=True)

# Menampilkan data setelah diurutkan
df.head(10)
# 3. Menentukan fitur input dan target
# Pada baseline LSTM ini digunakan pendekatan univariate time series:
#   - fitur (X) = nilai produksi EBT itu sendiri (lag/sliding window)
#   - target (y) = nilai produksi EBT pada bulan berikutnya
FEATURE_COL = "Produksi"
TARGET_COL = "Produksi"

# Mengambil daftar unik jenis PLT yang akan diproses satu per satu
daftar_plt = df["Jenis"].unique()
print("Jenis PLT yang akan diproses:", daftar_plt)
# [FIX LEAKAGE #1] WINDOW_SIZE dipindah ke sini (sebelumnya didefinisikan
# lebih bawah) supaya bisa dipakai untuk menghitung batas train/test SEBELUM
# scaler di-fit -- lihat audit/results/audit_findings.md bagian "Data
# Leakage (Scaler Fit-Before-Split)".
WINDOW_SIZE = 6

# [FIX SPLIT #1] VAL_INTERNAL_RATIO dipindah ke sini (sebelumnya hanya
# didefinisikan di tahap Fine-Tuning) supaya Baseline bisa memakai rasio
# validasi internal yang sama persis -- lihat Task Group 1 (perbaikan split
# Baseline jadi time-based) di catatan docstring atas file ini.
# TRAIN_RATIO (rasio index 80/20) DIHAPUS -- split Baseline sekarang
# time-based per tahun kalender, sama seperti Fine-Tuning/Iterasi 1-3.
VAL_INTERNAL_RATIO = 0.85

# 4. Normalisasi data menggunakan MinMaxScaler (dilakukan per jenis PLT)
# Dictionary untuk menyimpan data yang telah dinormalisasi per jenis PLT
data_scaled_per_plt = {}   # menyimpan hasil normalisasi (array)
scaler_per_plt = {}        # menyimpan objek scaler tiap PLT (dipakai untuk inverse transform)

for plt_name in daftar_plt:
    # Mengambil data untuk 1 jenis PLT saja, urut berdasarkan waktu
    data_plt = df[df["Jenis"] == plt_name][[FEATURE_COL]].values
    tanggal_plt = df[df["Jenis"] == plt_name]["Tanggal"].values

    # [FIX LEAKAGE #1] Sebelumnya: scaler.fit_transform(data_plt) -- di-fit
    # pada SELURUH seri (termasuk porsi yang nanti jadi data uji/test).
    # [FIX SPLIT #1] Batas fit scaler sekarang time-based (tahun <=2024 =
    # periode train+val), bukan lagi rasio index (TRAIN_RATIO) -- sejajar
    # dengan periode train/test yang dipakai dataset_per_plt di bawah dan
    # dengan pola scaler regional di "[FIX LEAKAGE #3]".
    train_val_mask_plt = pd.DatetimeIndex(tanggal_plt).year <= 2024

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(data_plt[train_val_mask_plt])
    data_plt_scaled = scaler.transform(data_plt)

    # Menyimpan hasil normalisasi, scaler, dan tanggal ke dictionary
    data_scaled_per_plt[plt_name] = data_plt_scaled
    scaler_per_plt[plt_name] = scaler
    data_scaled_per_plt[plt_name + "_tanggal"] = tanggal_plt

print("Normalisasi data selesai dilakukan untuk seluruh jenis PLT (scaler di-fit hanya pada porsi train+val 2023-2024, lihat FIX LEAKAGE #1 dan FIX SPLIT #1).")

def create_sequences(data, window_size=6):
    """
    Membuat data sequence (X, y) menggunakan pendekatan sliding window.

    Parameters
    ----------
    data : array-like
        Data time series yang sudah dinormalisasi, berbentuk (jumlah_data, 1)
    window_size : int
        Jumlah bulan yang digunakan sebagai input (default 6 bulan)

    Returns
    -------
    X : np.array, berbentuk (jumlah_sample, window_size, 1)
    y : np.array, berbentuk (jumlah_sample,)
    """
    X, y = [], []

    for i in range(len(data) - window_size):
        # Mengambil window_size data berturut-turut sebagai input
        X.append(data[i : i + window_size])
        # Mengambil 1 data setelah window sebagai target
        y.append(data[i + window_size])

    return np.array(X), np.array(y)

# WINDOW_SIZE sudah didefinisikan lebih awal (lihat FIX LEAKAGE #1 di atas).
print(f"Fungsi create_sequences() berhasil dibuat dengan window_size = {WINDOW_SIZE} bulan.")

# Dictionary untuk menyimpan hasil split data per jenis PLT
dataset_per_plt = {}

for plt_name in daftar_plt:
    # Membuat sequence (X, y) dari data yang sudah dinormalisasi
    data_plt_scaled = data_scaled_per_plt[plt_name]
    tanggal_plt = data_scaled_per_plt[plt_name + "_tanggal"]
    X_plt, y_plt = create_sequences(data_plt_scaled, window_size=WINDOW_SIZE)

    # [FIX SPLIT #1] Split train/val/test sekarang time-based per tahun
    # kalender (sama polanya dengan KEGIATAN 4 di tahap Fine-Tuning),
    # bukan lagi titik pemisah berbasis rasio index (TRAIN_RATIO).
    # Tanggal disesuaikan agar sejajar dengan titik target (bukan titik awal window).
    tanggal_target = pd.to_datetime(tanggal_plt[WINDOW_SIZE:])

    mask_train_val = (tanggal_target.year >= 2023) & (tanggal_target.year <= 2024)
    mask_test = tanggal_target.year == 2025

    X_train_val = X_plt[mask_train_val]
    y_train_val = y_plt[mask_train_val]
    X_test = X_plt[mask_test]
    y_test = y_plt[mask_test]

    # Split internal (time-based, tanpa acak) khusus untuk validasi selama training
    split_idx = int(len(X_train_val) * VAL_INTERNAL_RATIO)
    X_train = X_train_val[:split_idx]
    y_train = y_train_val[:split_idx]
    X_val = X_train_val[split_idx:]
    y_val = y_train_val[split_idx:]

    # Menyimpan hasil split ke dictionary agar mudah dipanggil kembali saat training
    dataset_per_plt[plt_name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
    }

    print(f"{plt_name:15s} -> Total sample: {len(X_plt):3d} | Train: {len(X_train):3d} | Val: {len(X_val):3d} | Test: {len(X_test):3d}")

def build_baseline_lstm(window_size, n_features=1, l2_rate=0.001, dropout_rate=0.2):
    """
    Membangun arsitektur Baseline LSTM sederhana.

    Parameters
    ----------
    window_size : int
        Panjang sliding window (jumlah time step input)
    n_features : int
        Jumlah fitur input (1 karena pendekatan univariate)
    l2_rate : float
        Nilai regularisasi L2 untuk mengurangi overfitting
    dropout_rate : float
        Proporsi neuron yang di-drop pada layer Dropout

    Returns
    -------
    model : tf.keras.Model
        Model LSTM yang sudah dikompilasi
    """
    model = Sequential([
        # Layer LSTM pertama, return_sequences=True karena masih akan diikuti LSTM lagi
        LSTM(64, activation="tanh", return_sequences=True,
             kernel_regularizer=l2(l2_rate),
             input_shape=(window_size, n_features)),
        Dropout(dropout_rate),

        # Layer LSTM kedua, return_sequences=False karena akan masuk ke Dense
        LSTM(32, activation="tanh", return_sequences=False,
             kernel_regularizer=l2(l2_rate)),
        Dropout(dropout_rate),

        # Layer Dense output, 1 neuron karena memprediksi 1 nilai produksi EBT
        Dense(1)
    ])

    # Compile model menggunakan optimizer Adam dan loss MSE (umum untuk regresi)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])

    return model

# Menampilkan contoh ringkasan arsitektur model
contoh_model = build_baseline_lstm(window_size=WINDOW_SIZE)
contoh_model.summary()

def get_callbacks():
    """
    Membuat daftar callback yang akan digunakan saat training model.
    Fungsi ini dipanggil ulang untuk setiap PLT agar callback selalu dalam kondisi baru (fresh).
    """
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=0
    )

    return [early_stop, reduce_lr]

print("Callback EarlyStopping dan ReduceLROnPlateau berhasil disiapkan.")

# Dictionary untuk menyimpan model dan history training tiap PLT (hanya di memori sesi ini)
hasil_training_per_plt = {}

EPOCHS = 100
BATCH_SIZE = 8

for plt_name in daftar_plt:
    print(f"\n=== Melatih Model Baseline LSTM untuk: {plt_name} ===")

    # Mengambil data train dan validasi internal untuk jenis PLT saat ini
    # [FIX SPLIT #1] Sebelumnya divalidasi dengan X_test/y_test (data uji
    # 2025) -- sekarang divalidasi dengan X_val/y_val (bagian akhir periode
    # train+val 2023-2024), supaya EarlyStopping/ReduceLROnPlateau tidak
    # pernah melihat data uji akhir sama sekali.
    X_train = dataset_per_plt[plt_name]["X_train"]
    y_train = dataset_per_plt[plt_name]["y_train"]
    X_val = dataset_per_plt[plt_name]["X_val"]
    y_val = dataset_per_plt[plt_name]["y_val"]

    # Membangun model baru (arsitektur sama untuk semua PLT)
    model = build_baseline_lstm(window_size=WINDOW_SIZE)

    # Melatih model menggunakan data train, divalidasi dengan data validasi internal
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks(),
        verbose=0
    )

    # Menyimpan model dan history ke dictionary (sementara, di memori)
    hasil_training_per_plt[plt_name] = {
        "model": model,
        "history": history
    }

    print(f"Training selesai. Epoch berhenti pada epoch ke-{len(history.history['loss'])}")

# Menampilkan grafik loss untuk setiap jenis PLT
n_plt = len(daftar_plt)
n_cols = 2
n_rows = int(np.ceil(n_plt / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt):
    history = hasil_training_per_plt[plt_name]["history"]

    axes[i].plot(history.history["loss"], label="Training Loss")
    axes[i].plot(history.history["val_loss"], label="Validation Loss")
    axes[i].set_title(f"Loss - {plt_name}")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Loss (MSE)")
    axes[i].legend()

# Menyembunyikan subplot kosong jika jumlah PLT ganjil
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Dictionary untuk menyimpan hasil prediksi (skala asli) per jenis PLT
hasil_prediksi_per_plt = {}

for plt_name in daftar_plt:
    model = hasil_training_per_plt[plt_name]["model"]
    scaler = scaler_per_plt[plt_name]

    X_test = dataset_per_plt[plt_name]["X_test"]
    y_test = dataset_per_plt[plt_name]["y_test"]

    # Melakukan prediksi pada data uji (masih dalam skala normalisasi 0-1)
    y_pred_scaled = model.predict(X_test, verbose=0)

    # Mengembalikan hasil prediksi dan data aktual ke skala asli (GWh)
    y_pred_asli = scaler.inverse_transform(y_pred_scaled)
    y_test_asli = scaler.inverse_transform(y_test.reshape(-1, 1))

    # Menyimpan hasil prediksi dan aktual (skala asli) ke dictionary
    hasil_prediksi_per_plt[plt_name] = {
        "y_test_asli": y_test_asli.flatten(),
        "y_pred_asli": y_pred_asli.flatten()
    }

    print(f"Prediksi untuk {plt_name} selesai. Jumlah data uji: {len(y_test_asli)}")

def hitung_mape(y_true, y_pred):
    """
    Menghitung Mean Absolute Percentage Error (MAPE).
    Nilai mendekati 0 pada y_true diabaikan untuk menghindari pembagian dengan nol.
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0  # menghindari pembagian dengan nol
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# Dictionary untuk menyimpan skor evaluasi tiap PLT
evaluasi_per_plt = {}

for plt_name in daftar_plt:
    y_test_asli = hasil_prediksi_per_plt[plt_name]["y_test_asli"]
    y_pred_asli = hasil_prediksi_per_plt[plt_name]["y_pred_asli"]

    rmse = np.sqrt(mean_squared_error(y_test_asli, y_pred_asli))
    mae = mean_absolute_error(y_test_asli, y_pred_asli)
    mape = hitung_mape(y_test_asli, y_pred_asli)

    evaluasi_per_plt[plt_name] = {"RMSE": rmse, "MAE": mae, "MAPE": mape}

    print(f"{plt_name:15s} -> RMSE: {rmse:8.3f} | MAE: {mae:8.3f} | MAPE: {mape:6.2f}%")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt):
    y_test_asli = hasil_prediksi_per_plt[plt_name]["y_test_asli"]
    y_pred_asli = hasil_prediksi_per_plt[plt_name]["y_pred_asli"]

    axes[i].plot(y_test_asli, label="Aktual", marker="o")
    axes[i].plot(y_pred_asli, label="Prediksi", marker="x")
    axes[i].set_title(f"Aktual vs Prediksi - {plt_name}")
    axes[i].set_xlabel("Periode (Data Uji)")
    axes[i].set_ylabel("Produksi EBT (GWh)")
    axes[i].legend()

# Menyembunyikan subplot kosong jika jumlah PLT ganjil
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Menggabungkan hasil evaluasi seluruh PLT ke dalam satu DataFrame
df_evaluasi = pd.DataFrame(evaluasi_per_plt).T  # transpose agar jenis PLT menjadi baris
df_evaluasi.index.name = "Jenis_PLT"
df_evaluasi = df_evaluasi.reset_index()

# Mengurutkan berdasarkan nilai RMSE terkecil (model terbaik) ke terbesar
df_evaluasi = df_evaluasi.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

In [ ]:
# ============================================================
# [TASK GROUP 2] Evaluasi Baseline pada DATA VALIDASI (X_val/y_val)
# Dipakai untuk PEMILIHAN model terbaik (lihat KEGIATAN 10 di bagian
# perbandingan seluruh model) -- RMSE test 2025 di atas TETAP dihitung
# dan dilaporkan, tapi bukan lagi dasar pemilihan (lihat audit/results
# untuk detail temuan test-set leakage pada pemilihan model).
# ============================================================

evaluasi_val_per_plt = {}

for plt_name in daftar_plt:
    model = hasil_training_per_plt[plt_name]["model"]
    scaler = scaler_per_plt[plt_name]

    X_val = dataset_per_plt[plt_name]["X_val"]
    y_val = dataset_per_plt[plt_name]["y_val"]

    y_pred_scaled_val = model.predict(X_val, verbose=0)
    y_pred_asli_val = scaler.inverse_transform(y_pred_scaled_val)
    y_val_asli = scaler.inverse_transform(y_val.reshape(-1, 1))

    rmse_val = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli_val))
    mae_val = mean_absolute_error(y_val_asli, y_pred_asli_val)
    mape_val = hitung_mape(y_val_asli, y_pred_asli_val)

    evaluasi_val_per_plt[plt_name] = {"RMSE": rmse_val, "MAE": mae_val, "MAPE": mape_val}

    print(f"{plt_name:15s} -> [VALIDASI] RMSE: {rmse_val:8.3f} | MAE: {mae_val:8.3f} | MAPE: {mape_val:6.2f}%")

df_evaluasi_val = pd.DataFrame(evaluasi_val_per_plt).T
df_evaluasi_val.index.name = "Jenis_PLT"
df_evaluasi_val = df_evaluasi_val.reset_index()
df_evaluasi_val = df_evaluasi_val.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("Ringkasan Hasil Evaluasi Model Baseline LSTM per Jenis PLT:")
df_evaluasi

In [ ]:
# =========================================================
# RMSE GABUNGAN (OVERALL) - BUKAN RATA-RATA ANTAR PLT
# =========================================================
# Tujuan: menghitung 1 nilai RMSE tunggal dari seluruh data uji
# (semua jenis PLT digabung), untuk dijadikan acuan pembanding
# utama terhadap hasil model Transfer Learning nantinya.

# Menggabungkan seluruh nilai aktual dan prediksi (skala asli) dari semua jenis PLT
y_test_gabungan = np.concatenate(
    [hasil_prediksi_per_plt[plt_name]["y_test_asli"] for plt_name in daftar_plt]
)
y_pred_gabungan = np.concatenate(
    [hasil_prediksi_per_plt[plt_name]["y_pred_asli"] for plt_name in daftar_plt]
)

# Menghitung RMSE gabungan (overall), bukan rata-rata RMSE per PLT
rmse_gabungan = np.sqrt(mean_squared_error(y_test_gabungan, y_pred_gabungan))

# Sebagai pembanding, ditampilkan juga rata-rata RMSE per PLT (dari df_evaluasi)
rmse_rata2_per_plt = df_evaluasi["RMSE"].mean()

print(f"Total data uji gabungan (seluruh PLT) : {len(y_test_gabungan)}")
print(f"RMSE Gabungan (Overall Baseline)      : {rmse_gabungan:.3f}")
print(f"Rata-rata RMSE per PLT (pembanding)   : {rmse_rata2_per_plt:.3f}")

PRE-TRAINING DATA NASIONAL

In [ ]:
import pandas as pd

In [ ]:
# ============================================================
# KEGIATAN 1: Memuat Dataset Nasional dan Eksplorasi Data
# ============================================================

# Path dataset nasional (2020-2024)
# Sesuaikan path dan nama file dengan dataset nasional yang sebenarnya digunakan
DATASET_NASIONAL_PATH = "DATA_PHASE_2_NASIONAL_FINAL.csv"

# Membaca dataset nasional
df_nasional = pd.read_csv(DATASET_NASIONAL_PATH)

# Menampilkan 5 baris pertama data
print("Contoh Data Nasional:")
display(df_nasional.head())

# Menampilkan informasi umum dataset (jumlah baris, tipe data, dsb)
print("\nInformasi Dataset Nasional:")
df_nasional.info()

# Menampilkan jumlah data per jenis PLT
print("\nJumlah Data per Jenis PLT (Nasional):")
print(df_nasional["Jenis"].value_counts())

# Mengecek missing value per kolom
print("\nJumlah Missing Value per Kolom:")
print(df_nasional.isnull().sum())

# Menampilkan statistik deskriptif dataset nasional
print("\nStatistik Deskriptif Dataset Nasional:")
display(df_nasional.describe(include="all"))

from sklearn.preprocessing import MinMaxScaler

In [ ]:
# ============================================================
# KEGIATAN 2: Preprocessing Data Nasional
# ============================================================

# 1. Mengonversi kolom tanggal menjadi tipe data datetime
df_nasional["Tanggal"] = pd.to_datetime(df_nasional["Tanggal"])

# 2. Mengurutkan data berdasarkan jenis PLT lalu berdasarkan waktu
df_nasional = df_nasional.sort_values(by=["Jenis", "Tanggal"]).reset_index(drop=True)

# 3. Menentukan fitur input dan target
#    Fitur input : Cuaca dan Kapasitas
#    Target      : Produksi
FEATURE_COLS_NASIONAL = ["Cuaca", "Kapasitas"]
TARGET_COL_NASIONAL = "Produksi"

# Daftar jenis PLT pada dataset nasional
daftar_plt_nasional = df_nasional["Jenis"].unique()
print("Jenis PLT pada dataset nasional:", daftar_plt_nasional)

# 4. Normalisasi data menggunakan MinMaxScaler KHUSUS data nasional
#    (scaler dibuat terpisah dari scaler Baseline regional, dan terpisah per jenis PLT)
scaler_fitur_nasional_per_plt = {}   # scaler untuk fitur input (Cuaca, Kapasitas)
scaler_target_nasional_per_plt = {}  # scaler untuk target (Produksi)
data_scaled_nasional_per_plt = {}    # menyimpan array gabungan (fitur+target) hasil normalisasi

for plt_name in daftar_plt_nasional:
    data_plt = df_nasional[df_nasional["Jenis"] == plt_name].copy()

    fitur_plt = data_plt[FEATURE_COLS_NASIONAL].values
    target_plt = data_plt[[TARGET_COL_NASIONAL]].values

    # Scaler terpisah untuk fitur dan target (target perlu discaler sendiri agar bisa di-inverse_transform)
    scaler_fitur = MinMaxScaler(feature_range=(0, 1))
    scaler_target = MinMaxScaler(feature_range=(0, 1))

    # [FIX LEAKAGE #2] Sebelumnya: fit_transform pada SELURUH data_plt
    # (2020-2024), termasuk tahun 2024 yang dipakai sebagai data validasi
    # (lihat KEGIATAN 4 di bawah: mask_train = tahun<=2023, mask_val =
    # tahun==2024). Sekarang: scaler HANYA di-fit dari baris tahun <=2023
    # (periode train), lalu dipakai transform ke seluruh data_plt.
    train_mask_plt = (data_plt["Tanggal"].dt.year <= 2023).values
    scaler_fitur.fit(fitur_plt[train_mask_plt])
    scaler_target.fit(target_plt[train_mask_plt])
    fitur_scaled = scaler_fitur.transform(fitur_plt)
    target_scaled = scaler_target.transform(target_plt)

    # Menggabungkan fitur dan target menjadi satu array,
    # dengan kolom target diletakkan PALING AKHIR agar mudah diambil kembali nanti
    data_gabungan_scaled = np.hstack([fitur_scaled, target_scaled])

    scaler_fitur_nasional_per_plt[plt_name] = scaler_fitur
    scaler_target_nasional_per_plt[plt_name] = scaler_target
    data_scaled_nasional_per_plt[plt_name] = data_gabungan_scaled

    # Menyimpan juga tanggal (untuk keperluan split berdasarkan tahun pada kegiatan berikutnya)
    data_scaled_nasional_per_plt[plt_name + "_tanggal"] = data_plt["Tanggal"].values

print("Normalisasi data nasional selesai dilakukan untuk seluruh jenis PLT.")

In [ ]:
# ============================================================
# KEGIATAN 3: Membentuk Data Time Series dengan create_sequences()
# (Fungsi create_sequences() TIDAK dibuat ulang, menggunakan yang sudah ada)
# ============================================================

# Dictionary untuk menyimpan hasil sequence (X, y) beserta tanggalnya per jenis PLT
sequence_nasional_per_plt = {}

for plt_name in daftar_plt_nasional:
    data_gabungan_scaled = data_scaled_nasional_per_plt[plt_name]
    tanggal_plt = data_scaled_nasional_per_plt[plt_name + "_tanggal"]

    # Menggunakan kembali create_sequences() yang sudah dibuat pada tahap Baseline
    # Karena data di sini multi-kolom (Cuaca, Kapasitas, Produksi), maka:
    #   X_seq -> berisi seluruh kolom (fitur + target) selama window_size bulan
    #   y_seq -> berisi seluruh kolom pada 1 bulan setelah window, target diambil dari kolom terakhir
    X_seq, y_seq = create_sequences(data_gabungan_scaled, window_size=WINDOW_SIZE)

    # Mengambil hanya kolom target (Produksi) dari y_seq, yaitu kolom paling akhir
    y_target_seq = y_seq[:, -1]

    # Menyesuaikan tanggal agar sejajar dengan y (tanggal pada titik target, bukan titik awal window)
    tanggal_target = tanggal_plt[WINDOW_SIZE:]

    sequence_nasional_per_plt[plt_name] = {
        "X": X_seq,                 # shape: (jumlah_sample, window_size, jumlah_fitur)
        "y": y_target_seq,          # shape: (jumlah_sample,)
        "tanggal_target": tanggal_target
    }

    print(f"{plt_name:8s} -> Total sequence terbentuk: {len(X_seq)}")

In [ ]:
# ============================================================
# KEGIATAN 4: Split Data Berdasarkan Waktu (Tanpa Pengacakan)
# Train (Pre-Training) : 2020-2023
# Validasi              : 2024
# ============================================================

dataset_pretrain_per_plt = {}

for plt_name in daftar_plt_nasional:
    X_seq = sequence_nasional_per_plt[plt_name]["X"]
    y_seq = sequence_nasional_per_plt[plt_name]["y"]
    tanggal_target = pd.to_datetime(sequence_nasional_per_plt[plt_name]["tanggal_target"])

    # Membuat mask berdasarkan tahun pada tanggal target (bukan tanggal awal window)
    mask_train = tanggal_target.year <= 2023
    mask_val = tanggal_target.year == 2024

    X_train = X_seq[mask_train]
    y_train = y_seq[mask_train]
    X_val = X_seq[mask_val]
    y_val = y_seq[mask_val]

    dataset_pretrain_per_plt[plt_name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
    }

    print(f"{plt_name:8s} -> Train (2020-2023): {len(X_train):3d} | Validasi (2024): {len(X_val):3d}")

In [ ]:
# ============================================================
# KEGIATAN 5: Menggunakan Kembali Arsitektur Baseline LSTM
# (TIDAK ADA perubahan layer, neuron, dropout, maupun regularisasi)
# ============================================================

# Fungsi build_baseline_lstm() sudah dibuat pada tahap Baseline dan digunakan kembali di sini.
# Yang berbeda hanya jumlah fitur input (n_features), karena data nasional bersifat multivariate
# (Cuaca, Kapasitas, Produksi), sementara arsitekturnya tetap identik dengan Baseline.

N_FEATURES_NASIONAL = len(FEATURE_COLS_NASIONAL) + 1  # +1 karena Produksi historis juga ikut sebagai fitur

contoh_model_pretrain = build_baseline_lstm(window_size=WINDOW_SIZE, n_features=N_FEATURES_NASIONAL)
contoh_model_pretrain.summary()

In [ ]:
# ============================================================
# KEGIATAN 6: Menyiapkan Callback (Sama Seperti Baseline)
# ============================================================

# Fungsi get_callbacks() sudah dibuat pada tahap Baseline, digunakan kembali tanpa perubahan.
print("Callback EarlyStopping dan ReduceLROnPlateau siap digunakan untuk proses pre-training.")

In [ ]:
# ============================================================
# KEGIATAN 7: Pre-Training Model Secara Terpisah per Jenis PLT
# (PLTA, PLTB, PLTM, PLTMH, PLTS)
# ============================================================

hasil_pretrain_per_plt = {}

EPOCHS_PRETRAIN = 100
BATCH_SIZE_PRETRAIN = 16  # batch size lebih besar karena data nasional lebih banyak

for plt_name in daftar_plt_nasional:
    print(f"\n=== Pre-Training Model untuk: {plt_name} ===")

    X_train = dataset_pretrain_per_plt[plt_name]["X_train"]
    y_train = dataset_pretrain_per_plt[plt_name]["y_train"]
    X_val = dataset_pretrain_per_plt[plt_name]["X_val"]
    y_val = dataset_pretrain_per_plt[plt_name]["y_val"]

    # Membangun model baru dengan arsitektur yang sama persis seperti Baseline
    model_pretrain = build_baseline_lstm(window_size=WINDOW_SIZE, n_features=N_FEATURES_NASIONAL)

    history_pretrain = model_pretrain.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_PRETRAIN,
        batch_size=BATCH_SIZE_PRETRAIN,
        callbacks=get_callbacks(),
        verbose=0
    )

    hasil_pretrain_per_plt[plt_name] = {
        "model": model_pretrain,
        "history": history_pretrain
    }

    print(f"Pre-training selesai. Epoch berhenti pada epoch ke-{len(history_pretrain.history['loss'])}")

In [ ]:
# ============================================================
# KEGIATAN 8: Menampilkan Grafik Training Loss dan Validation Loss
# ============================================================

n_plt_nasional = len(daftar_plt_nasional)
n_cols = 2
n_rows = int(np.ceil(n_plt_nasional / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_nasional):
    history_pretrain = hasil_pretrain_per_plt[plt_name]["history"]

    axes[i].plot(history_pretrain.history["loss"], label="Training Loss")
    axes[i].plot(history_pretrain.history["val_loss"], label="Validation Loss")
    axes[i].set_title(f"Loss Pre-Training - {plt_name}")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Loss (MSE)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 9: Prediksi pada Data Validasi dan Inverse Transform
# ============================================================

hasil_prediksi_pretrain_per_plt = {}

for plt_name in daftar_plt_nasional:
    model_pretrain = hasil_pretrain_per_plt[plt_name]["model"]
    scaler_target = scaler_target_nasional_per_plt[plt_name]

    X_val = dataset_pretrain_per_plt[plt_name]["X_val"]
    y_val = dataset_pretrain_per_plt[plt_name]["y_val"]

    # Prediksi pada data validasi (masih dalam skala normalisasi)
    y_pred_scaled = model_pretrain.predict(X_val, verbose=0)

    # Mengembalikan hasil prediksi dan data aktual ke skala asli menggunakan scaler_target
    y_pred_asli = scaler_target.inverse_transform(y_pred_scaled)
    y_val_asli = scaler_target.inverse_transform(y_val.reshape(-1, 1))

    hasil_prediksi_pretrain_per_plt[plt_name] = {
        "y_val_asli": y_val_asli.flatten(),
        "y_pred_asli": y_pred_asli.flatten()
    }

    print(f"Prediksi untuk {plt_name} selesai. Jumlah data validasi: {len(y_val_asli)}")

In [ ]:
# ============================================================
# KEGIATAN 10: Evaluasi Model Menggunakan RMSE, MAE, dan MAPE
# ============================================================

# Fungsi hitung_mape() sudah dibuat pada tahap Baseline, digunakan kembali di sini.
evaluasi_pretrain_per_plt = {}

for plt_name in daftar_plt_nasional:
    y_val_asli = hasil_prediksi_pretrain_per_plt[plt_name]["y_val_asli"]
    y_pred_asli = hasil_prediksi_pretrain_per_plt[plt_name]["y_pred_asli"]

    rmse = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli))
    mae = mean_absolute_error(y_val_asli, y_pred_asli)
    mape = hitung_mape(y_val_asli, y_pred_asli)

    evaluasi_pretrain_per_plt[plt_name] = {"RMSE": rmse, "MAE": mae, "MAPE": mape}

    print(f"{plt_name:8s} -> RMSE: {rmse:8.3f} | MAE: {mae:8.3f} | MAPE: {mape:6.2f}%")

In [ ]:
# ============================================================
# KEGIATAN 11: Memvisualisasikan Perbandingan Data Aktual dan Prediksi
# ============================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_nasional):
    y_val_asli = hasil_prediksi_pretrain_per_plt[plt_name]["y_val_asli"]
    y_pred_asli = hasil_prediksi_pretrain_per_plt[plt_name]["y_pred_asli"]

    axes[i].plot(y_val_asli, label="Aktual", marker="o")
    axes[i].plot(y_pred_asli, label="Prediksi", marker="x")
    axes[i].set_title(f"Aktual vs Prediksi (Validasi 2024) - {plt_name}")
    axes[i].set_xlabel("Periode (Data Validasi)")
    axes[i].set_ylabel("Produksi EBT (GWh)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 12: Menggabungkan Seluruh Hasil Evaluasi Pre-Training
# ke Dalam Satu DataFrame (Diurutkan Berdasarkan RMSE)
# ============================================================

df_evaluasi_pretrain = pd.DataFrame(evaluasi_pretrain_per_plt).T
df_evaluasi_pretrain.index.name = "Jenis_PLT"
df_evaluasi_pretrain = df_evaluasi_pretrain.reset_index()

df_evaluasi_pretrain = df_evaluasi_pretrain.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("Ringkasan Hasil Evaluasi Pre-Training (Data Nasional) per Jenis PLT:")
df_evaluasi_pretrain

In [ ]:
# ============================================================
# KEGIATAN 1: Memuat Dataset Regional Sulawesi Selatan dan Eksplorasi Data
# ============================================================

# Path dataset regional Sulawesi Selatan (versi lengkap dengan fitur Cuaca & Kapasitas)
# Sesuaikan dengan dataset regional final yang digunakan pada penelitian
DATASET_REGIONAL_FT_PATH = "DATA_PHASE_3_REGIONAL_MODIFIED.csv"

df_regional = pd.read_csv(DATASET_REGIONAL_FT_PATH)

# Menampilkan 5 baris pertama data
print("Contoh Data Regional:")
display(df_regional.head())

# Informasi umum dataset
print("\nInformasi Dataset Regional:")
df_regional.info()

# Jumlah data per jenis PLT
print("\nJumlah Data per Jenis PLT (Regional):")
print(df_regional["Jenis"].value_counts())

# Missing value per kolom
print("\nJumlah Missing Value per Kolom:")
print(df_regional.isnull().sum())

# Statistik deskriptif
print("\nStatistik Deskriptif Dataset Regional:")
display(df_regional.describe(include="all"))

In [ ]:
# ============================================================
# KEGIATAN 2: Preprocessing Data Regional (Konsisten dengan Baseline & Pre-Training)
# ============================================================

# 1. Mengonversi kolom tanggal menjadi datetime
df_regional["Tanggal"] = pd.to_datetime(df_regional["Tanggal"])

# 2. Mengurutkan data berdasarkan jenis PLT lalu waktu
df_regional = df_regional.sort_values(by=["Jenis", "Tanggal"]).reset_index(drop=True)

# 3. Fitur input (Cuaca, Kapasitas) dan target (Produksi) — sama seperti Pre-Training
FEATURE_COLS_REGIONAL = ["Cuaca", "Kapasitas"]
TARGET_COL_REGIONAL = "Produksi"

daftar_plt_regional = df_regional["Jenis"].unique()
print("Jenis PLT pada dataset regional:", daftar_plt_regional)

# 4. Normalisasi MENGGUNAKAN MinMaxScaler KHUSUS data regional (scaler baru, terpisah per PLT)
scaler_fitur_regional_per_plt = {}
scaler_target_regional_per_plt = {}
data_scaled_regional_per_plt = {}

for plt_name in daftar_plt_regional:
    data_plt = df_regional[df_regional["Jenis"] == plt_name].copy()

    fitur_plt = data_plt[FEATURE_COLS_REGIONAL].values
    target_plt = data_plt[[TARGET_COL_REGIONAL]].values

    scaler_fitur = MinMaxScaler(feature_range=(0, 1))
    scaler_target = MinMaxScaler(feature_range=(0, 1))

    # [FIX LEAKAGE #3] Sebelumnya: fit_transform pada SELURUH data_plt
    # (2023-2025), termasuk tahun 2025 yang dipakai sebagai data uji akhir
    # (lihat KEGIATAN 4 di bawah: mask_finetune = 2023-2024, mask_test =
    # 2025). Scaler ini dipakai bersama oleh tahap Fine-Tuning DAN Iterasi
    # 1/2/3 (mereka reuse data_scaled_regional_per_plt, tidak fit ulang),
    # jadi perbaikan ini otomatis berlaku untuk keempat tahap sekaligus.
    # (Scaler terpisah punya "Model Final" -- KEGIATAN 2 di bawah -- yang
    # SENGAJA fit dari seluruh 2023-2025 karena tahap itu tidak menyisakan
    # data uji sama sekali, jadi tidak termasuk perbaikan ini.)
    train_mask_plt = (data_plt["Tanggal"].dt.year <= 2024).values
    scaler_fitur.fit(fitur_plt[train_mask_plt])
    scaler_target.fit(target_plt[train_mask_plt])
    fitur_scaled = scaler_fitur.transform(fitur_plt)
    target_scaled = scaler_target.transform(target_plt)

    # Target diletakkan di kolom paling akhir (konsisten dengan Pre-Training)
    data_gabungan_scaled = np.hstack([fitur_scaled, target_scaled])

    scaler_fitur_regional_per_plt[plt_name] = scaler_fitur
    scaler_target_regional_per_plt[plt_name] = scaler_target
    data_scaled_regional_per_plt[plt_name] = data_gabungan_scaled
    data_scaled_regional_per_plt[plt_name + "_tanggal"] = data_plt["Tanggal"].values

print("Normalisasi data regional selesai dilakukan untuk seluruh jenis PLT.")

In [ ]:
# ============================================================
# KEGIATAN 3: Membentuk Data Time Series dengan create_sequences()
# (Fungsi TIDAK dibuat ulang, menggunakan yang sudah ada dari Baseline)
# ============================================================

sequence_regional_per_plt = {}

for plt_name in daftar_plt_regional:
    data_gabungan_scaled = data_scaled_regional_per_plt[plt_name]
    tanggal_plt = data_scaled_regional_per_plt[plt_name + "_tanggal"]

    X_seq, y_seq = create_sequences(data_gabungan_scaled, window_size=WINDOW_SIZE)

    # Mengambil kolom target (Produksi) saja, yaitu kolom paling akhir
    y_target_seq = y_seq[:, -1]

    # Tanggal disesuaikan agar sejajar dengan titik target (bukan titik awal window)
    tanggal_target = tanggal_plt[WINDOW_SIZE:]

    sequence_regional_per_plt[plt_name] = {
        "X": X_seq,
        "y": y_target_seq,
        "tanggal_target": tanggal_target
    }

    print(f"{plt_name:12s} -> Total sequence terbentuk: {len(X_seq)}")

In [ ]:
# ============================================================
# KEGIATAN 4: Split Data Berdasarkan Waktu (Tanpa Pengacakan)
# Fine-Tuning : 2023-2024   |   Test Akhir : 2025
# ============================================================

dataset_finetune_per_plt = {}

# Rasio kecil dari periode Fine-Tuning (2023-2024) disisihkan sebagai VALIDASI INTERNAL
# untuk keperluan EarlyStopping/ReduceLROnPlateau, TANPA menyentuh data test 2025 sama sekali.
# VAL_INTERNAL_RATIO sudah didefinisikan di awal file (dekat WINDOW_SIZE, lihat
# "[FIX SPLIT #1]") supaya Baseline bisa memakai rasio yang sama persis.

for plt_name in daftar_plt_regional:
    X_seq = sequence_regional_per_plt[plt_name]["X"]
    y_seq = sequence_regional_per_plt[plt_name]["y"]
    tanggal_target = pd.to_datetime(sequence_regional_per_plt[plt_name]["tanggal_target"])

    # Mask periode Fine-Tuning (2023-2024) dan Test akhir (2025)
    mask_finetune = (tanggal_target.year >= 2023) & (tanggal_target.year <= 2024)
    mask_test = tanggal_target.year == 2025

    X_finetune = X_seq[mask_finetune]
    y_finetune = y_seq[mask_finetune]
    X_test = X_seq[mask_test]
    y_test = y_seq[mask_test]

    # Split internal (time-based, tanpa acak) khusus untuk validasi selama Fine-Tuning
    split_idx = int(len(X_finetune) * VAL_INTERNAL_RATIO)
    X_train_ft = X_finetune[:split_idx]
    y_train_ft = y_finetune[:split_idx]
    X_val_ft = X_finetune[split_idx:]
    y_val_ft = y_finetune[split_idx:]

    dataset_finetune_per_plt[plt_name] = {
        "X_train_ft": X_train_ft,
        "y_train_ft": y_train_ft,
        "X_val_ft": X_val_ft,
        "y_val_ft": y_val_ft,
        "X_test": X_test,
        "y_test": y_test,
    }

    print(f"{plt_name:12s} -> Train FT: {len(X_train_ft):3d} | Val FT: {len(X_val_ft):3d} | Test 2025: {len(X_test):3d}")

In [ ]:
# ============================================================
# KEGIATAN 5: Fine-Tuning dengan Bobot Pre-Training sebagai Bobot Awal
# Berlaku untuk: PLTA, PLTB, PLTM, PLTMH, PLTS
# ============================================================

from tensorflow.keras.optimizers import Adam  # import baru: dibutuhkan untuk mengatur learning rate kecil

# Daftar PLT yang memiliki hasil Pre-Training pada data nasional
PLT_TRANSFER_LEARNING = ["PLTA", "PLTB", "PLTM", "PLTMH", "PLTS"]

# Jumlah fitur regional HARUS SAMA dengan jumlah fitur nasional agar bobot bisa dimuat
N_FEATURES_REGIONAL = len(FEATURE_COLS_REGIONAL) + 1  # Cuaca, Kapasitas, Produksi historis

LR_FINETUNE = 0.0001  # learning rate lebih kecil dibanding Pre-Training agar pengetahuan awal tetap terjaga
EPOCHS_FINETUNE = 100
BATCH_SIZE_FINETUNE = 8

hasil_finetune_per_plt = {}

for plt_name in PLT_TRANSFER_LEARNING:
    print(f"\n=== Fine-Tuning (Transfer Learning) untuk: {plt_name} ===")

    X_train_ft = dataset_finetune_per_plt[plt_name]["X_train_ft"]
    y_train_ft = dataset_finetune_per_plt[plt_name]["y_train_ft"]
    X_val_ft = dataset_finetune_per_plt[plt_name]["X_val_ft"]
    y_val_ft = dataset_finetune_per_plt[plt_name]["y_val_ft"]

    # 1. Membangun ulang arsitektur yang SAMA PERSIS dengan Baseline/Pre-Training
    model_ft = build_baseline_lstm(window_size=WINDOW_SIZE, n_features=N_FEATURES_REGIONAL)

    # 2. Memuat bobot hasil Pre-Training sebagai bobot awal (initial weights)
    bobot_pretrain = hasil_pretrain_per_plt[plt_name]["model"].get_weights()
    model_ft.set_weights(bobot_pretrain)

    # 3. Meng-compile ulang model dengan learning rate yang lebih kecil untuk Fine-Tuning
    model_ft.compile(optimizer=Adam(learning_rate=LR_FINETUNE), loss="mse", metrics=["mae"])

    # 4. Melakukan Fine-Tuning menggunakan data regional (callback direuse dari Baseline)
    history_ft = model_ft.fit(
        X_train_ft, y_train_ft,
        validation_data=(X_val_ft, y_val_ft),
        epochs=EPOCHS_FINETUNE,
        batch_size=BATCH_SIZE_FINETUNE,
        callbacks=get_callbacks(),
        verbose=0
    )

    hasil_finetune_per_plt[plt_name] = {
        "model": model_ft,
        "history": history_ft,
        "metode": "Transfer Learning"
    }

    print(f"Fine-Tuning selesai. Epoch berhenti pada epoch ke-{len(history_ft.history['loss'])}")

In [ ]:
# ============================================================
# KEGIATAN 6: Direct Training untuk PLT Tanpa Data Nasional
# Berlaku untuk: PLTS Atap, PLT Hybrid (TIDAK memuat bobot Pre-Training)
# ============================================================

PLT_DIRECT_TRAINING = ["PLTS Atap", "PLT Hybrid"]

for plt_name in PLT_DIRECT_TRAINING:
    print(f"\n=== Direct Training (Tanpa Transfer Learning) untuk: {plt_name} ===")

    X_train_ft = dataset_finetune_per_plt[plt_name]["X_train_ft"]
    y_train_ft = dataset_finetune_per_plt[plt_name]["y_train_ft"]
    X_val_ft = dataset_finetune_per_plt[plt_name]["X_val_ft"]
    y_val_ft = dataset_finetune_per_plt[plt_name]["y_val_ft"]

    # Membangun arsitektur yang SAMA seperti Baseline, TANPA memuat bobot Pre-Training
    model_direct = build_baseline_lstm(window_size=WINDOW_SIZE, n_features=N_FEATURES_REGIONAL)
    # Menggunakan compile default (sama seperti Baseline), bukan learning rate kecil,
    # karena model ini belajar dari nol (bukan melanjutkan pengetahuan nasional)

    history_direct = model_direct.fit(
        X_train_ft, y_train_ft,
        validation_data=(X_val_ft, y_val_ft),
        epochs=EPOCHS_FINETUNE,
        batch_size=BATCH_SIZE_FINETUNE,
        callbacks=get_callbacks(),
        verbose=0
    )

    hasil_finetune_per_plt[plt_name] = {
        "model": model_direct,
        "history": history_direct,
        "metode": "Direct Training"
    }

    print(f"Direct Training selesai. Epoch berhenti pada epoch ke-{len(history_direct.history['loss'])}")

In [ ]:
# ============================================================
# KEGIATAN 7: Callback EarlyStopping & ReduceLROnPlateau
# (Fungsi get_callbacks() sudah dibuat pada tahap Baseline dan
#  telah digunakan langsung pada proses Fine-Tuning maupun Direct Training di atas)
# ============================================================

print("Callback EarlyStopping dan ReduceLROnPlateau digunakan untuk seluruh proses training pada Tahap 2.")

In [ ]:
# ============================================================
# KEGIATAN 8: Menampilkan Grafik Training Loss dan Validation Loss
# ============================================================

daftar_plt_final = PLT_TRANSFER_LEARNING + PLT_DIRECT_TRAINING
n_plt_final = len(daftar_plt_final)
n_cols = 2
n_rows = int(np.ceil(n_plt_final / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_final):
    history = hasil_finetune_per_plt[plt_name]["history"]
    metode = hasil_finetune_per_plt[plt_name]["metode"]

    axes[i].plot(history.history["loss"], label="Training Loss")
    axes[i].plot(history.history["val_loss"], label="Validation Loss")
    axes[i].set_title(f"Loss - {plt_name} ({metode})")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Loss (MSE)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 9: Prediksi pada Data Uji 2025 dan Inverse Transform
# ============================================================

hasil_prediksi_finetune_per_plt = {}

for plt_name in daftar_plt_final:
    model = hasil_finetune_per_plt[plt_name]["model"]
    scaler_target = scaler_target_regional_per_plt[plt_name]

    X_test = dataset_finetune_per_plt[plt_name]["X_test"]
    y_test = dataset_finetune_per_plt[plt_name]["y_test"]

    y_pred_scaled = model.predict(X_test, verbose=0)

    y_pred_asli = scaler_target.inverse_transform(y_pred_scaled)
    y_test_asli = scaler_target.inverse_transform(y_test.reshape(-1, 1))

    hasil_prediksi_finetune_per_plt[plt_name] = {
        "y_test_asli": y_test_asli.flatten(),
        "y_pred_asli": y_pred_asli.flatten()
    }

    print(f"Prediksi untuk {plt_name} selesai. Jumlah data uji (2025): {len(y_test_asli)}")

In [ ]:
# ============================================================
# KEGIATAN 10: Evaluasi Model Menggunakan RMSE, MAE, dan MAPE
# ============================================================

evaluasi_finetune_per_plt = {}

for plt_name in daftar_plt_final:
    y_test_asli = hasil_prediksi_finetune_per_plt[plt_name]["y_test_asli"]
    y_pred_asli = hasil_prediksi_finetune_per_plt[plt_name]["y_pred_asli"]

    rmse = np.sqrt(mean_squared_error(y_test_asli, y_pred_asli))
    mae = mean_absolute_error(y_test_asli, y_pred_asli)
    mape = hitung_mape(y_test_asli, y_pred_asli)

    evaluasi_finetune_per_plt[plt_name] = {"RMSE": rmse, "MAE": mae, "MAPE": mape}

    print(f"{plt_name:12s} -> RMSE: {rmse:8.3f} | MAE: {mae:8.3f} | MAPE: {mape:6.2f}%")

In [ ]:
# ============================================================
# KEGIATAN 11: Memvisualisasikan Perbandingan Data Aktual dan Prediksi (2025)
# ============================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_final):
    y_test_asli = hasil_prediksi_finetune_per_plt[plt_name]["y_test_asli"]
    y_pred_asli = hasil_prediksi_finetune_per_plt[plt_name]["y_pred_asli"]
    metode = hasil_finetune_per_plt[plt_name]["metode"]

    axes[i].plot(y_test_asli, label="Aktual", marker="o")
    axes[i].plot(y_pred_asli, label="Prediksi", marker="x")
    axes[i].set_title(f"Aktual vs Prediksi (Test 2025) - {plt_name} ({metode})")
    axes[i].set_xlabel("Periode (Data Uji 2025)")
    axes[i].set_ylabel("Produksi EBT (GWh)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 12: Menggabungkan Hasil Evaluasi ke DataFrame
# Diurutkan Berdasarkan RMSE, Disertai Kolom Metode (Transfer Learning / Direct Training)
# ============================================================

df_evaluasi_finetune = pd.DataFrame(evaluasi_finetune_per_plt).T
df_evaluasi_finetune.index.name = "Jenis_PLT"
df_evaluasi_finetune = df_evaluasi_finetune.reset_index()

# Menambahkan kolom Metode berdasarkan jenis PLT
df_evaluasi_finetune["Metode"] = df_evaluasi_finetune["Jenis_PLT"].apply(
    lambda plt_name: hasil_finetune_per_plt[plt_name]["metode"]
)

df_evaluasi_finetune = df_evaluasi_finetune.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("Ringkasan Hasil Evaluasi Fine-Tuning (Data Regional 2025) per Jenis PLT:")
df_evaluasi_finetune

In [ ]:
# ============================================================
# [TASK GROUP 2] Evaluasi Fine-Tuning pada DATA VALIDASI (X_val_ft/y_val_ft)
# Dipakai untuk PEMILIHAN model terbaik -- lihat catatan yang sama di
# tahap Baseline di atas.
# ============================================================

evaluasi_finetune_val_per_plt = {}

for plt_name in daftar_plt_final:
    model = hasil_finetune_per_plt[plt_name]["model"]
    scaler_target = scaler_target_regional_per_plt[plt_name]

    X_val_ft = dataset_finetune_per_plt[plt_name]["X_val_ft"]
    y_val_ft = dataset_finetune_per_plt[plt_name]["y_val_ft"]

    y_pred_scaled_val = model.predict(X_val_ft, verbose=0)
    y_pred_asli_val = scaler_target.inverse_transform(y_pred_scaled_val)
    y_val_asli = scaler_target.inverse_transform(y_val_ft.reshape(-1, 1))

    rmse_val = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli_val))
    mae_val = mean_absolute_error(y_val_asli, y_pred_asli_val)
    mape_val = hitung_mape(y_val_asli, y_pred_asli_val)

    evaluasi_finetune_val_per_plt[plt_name] = {"RMSE": rmse_val, "MAE": mae_val, "MAPE": mape_val}

    print(f"{plt_name:12s} -> [VALIDASI] RMSE: {rmse_val:8.3f} | MAE: {mae_val:8.3f} | MAPE: {mape_val:6.2f}%")

df_evaluasi_finetune_val = pd.DataFrame(evaluasi_finetune_val_per_plt).T
df_evaluasi_finetune_val.index.name = "Jenis_PLT"
df_evaluasi_finetune_val = df_evaluasi_finetune_val.reset_index()
df_evaluasi_finetune_val = df_evaluasi_finetune_val.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

In [ ]:
# ============================================================
# KEGIATAN 13: Tabel Perbandingan Baseline LSTM vs Fine-Tuning Transfer Learning
# ============================================================

# Menggabungkan hasil Baseline (df_evaluasi) dan Fine-Tuning (df_evaluasi_finetune)
# berdasarkan Jenis_PLT. Catatan: PLTS Atap dan PLT Hybrid tidak memiliki hasil Baseline
# jika sebelumnya belum tersedia pada dataset Baseline (akan tampil NaN pada RMSE_Baseline).
df_perbandingan = pd.merge(
    df_evaluasi[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
        columns={"RMSE": "RMSE_Baseline", "MAE": "MAE_Baseline", "MAPE": "MAPE_Baseline"}
    ),
    df_evaluasi_finetune[["Jenis_PLT", "RMSE", "MAE", "MAPE", "Metode"]].rename(
        columns={"RMSE": "RMSE_FineTuning", "MAE": "MAE_FineTuning", "MAPE": "MAPE_FineTuning"}
    ),
    on="Jenis_PLT",
    how="outer"
)

# Menghitung persentase penurunan RMSE (peningkatan performa) akibat Fine-Tuning
df_perbandingan["Penurunan_RMSE (%)"] = (
    (df_perbandingan["RMSE_Baseline"] - df_perbandingan["RMSE_FineTuning"])
    / df_perbandingan["RMSE_Baseline"] * 100
)

df_perbandingan = df_perbandingan.sort_values(by="RMSE_FineTuning", ascending=True).reset_index(drop=True)

print("Tabel Perbandingan Baseline LSTM vs Fine-Tuning (Transfer Learning) per Jenis PLT:")
df_perbandingan

In [ ]:
# ============================================================
# KEGIATAN 1: Menentukan Learning Rate Baru untuk Iterasi 1
# (Satu-satunya hyperparameter yang diubah pada iterasi ini)
# ============================================================

# Learning rate Fine-Tuning sebelumnya: 0.0001
# Learning rate Iterasi 1 dibuat lebih kecil agar penyesuaian bobot lebih halus
LR_ITER1 = 0.00005

# Parameter lain TETAP SAMA seperti Fine-Tuning (tidak diubah)
EPOCHS_ITER1 = EPOCHS_FINETUNE
BATCH_SIZE_ITER1 = BATCH_SIZE_FINETUNE

print(f"Learning rate Fine-Tuning : {LR_FINETUNE}")
print(f"Learning rate Iterasi 1   : {LR_ITER1}")

In [ ]:
# ============================================================
# KEGIATAN 2: Menggunakan Kembali Dataset Regional yang Sudah Dipreprocessing
# (Data, scaler, dan split train/val/test SAMA PERSIS dengan tahap Fine-Tuning)
# ============================================================

# dataset_finetune_per_plt sudah berisi X_train_ft, y_train_ft, X_val_ft, y_val_ft, X_test, y_test
# untuk seluruh jenis PLT (hasil split 2023-2024 training, 2025 testing).
# Tidak perlu load ulang dataset, preprocessing, maupun scaler.

print("Dataset regional (hasil preprocessing Fine-Tuning) digunakan kembali untuk Iterasi 1.")
print("Jenis PLT yang tersedia:", daftar_plt_final)

In [ ]:
# ============================================================
# KEGIATAN 3: Fungsi create_sequences() Digunakan Kembali
# (Sequence sudah terbentuk pada dataset_finetune_per_plt, tidak dibuat ulang)
# ============================================================

print(f"Sliding window yang digunakan tetap konsisten: WINDOW_SIZE = {WINDOW_SIZE} bulan.")

In [ ]:
# ============================================================
# KEGIATAN 4: Split Data Regional (Reuse) - Training 2023-2024, Testing 2025
# ============================================================

for plt_name in daftar_plt_final:
    n_train = len(dataset_finetune_per_plt[plt_name]["X_train_ft"])
    n_val = len(dataset_finetune_per_plt[plt_name]["X_val_ft"])
    n_test = len(dataset_finetune_per_plt[plt_name]["X_test"])
    print(f"{plt_name:12s} -> Train: {n_train:3d} | Val: {n_val:3d} | Test 2025: {n_test:3d}")

In [ ]:
# ============================================================
# KEGIATAN 5: Iterasi 1 - Transfer Learning dengan Learning Rate Baru
# Berlaku untuk: PLTA, PLTB, PLTM, PLTMH, PLTS
# (Arsitektur, bobot awal Pre-Training, dan data SAMA seperti Fine-Tuning;
#  yang berbeda HANYA learning rate)
# ============================================================

hasil_iter1_per_plt = {}

for plt_name in PLT_TRANSFER_LEARNING:
    print(f"\n=== Iterasi 1 (Transfer Learning) untuk: {plt_name} ===")

    X_train_ft = dataset_finetune_per_plt[plt_name]["X_train_ft"]
    y_train_ft = dataset_finetune_per_plt[plt_name]["y_train_ft"]
    X_val_ft = dataset_finetune_per_plt[plt_name]["X_val_ft"]
    y_val_ft = dataset_finetune_per_plt[plt_name]["y_val_ft"]

    # 1. Membangun ulang arsitektur yang SAMA PERSIS seperti Baseline/Fine-Tuning
    model_iter1 = build_baseline_lstm(window_size=WINDOW_SIZE, n_features=N_FEATURES_REGIONAL)

    # 2. Memuat bobot hasil Pre-Training sebagai bobot awal (sama seperti Fine-Tuning)
    bobot_pretrain = hasil_pretrain_per_plt[plt_name]["model"].get_weights()
    model_iter1.set_weights(bobot_pretrain)

    # 3. Compile ulang dengan learning rate Iterasi 1 (LR_ITER1), lebih kecil dari Fine-Tuning
    model_iter1.compile(optimizer=Adam(learning_rate=LR_ITER1), loss="mse", metrics=["mae"])

    # 4. Training menggunakan parameter lain yang sama seperti Fine-Tuning (epoch, batch size, callback)
    history_iter1 = model_iter1.fit(
        X_train_ft, y_train_ft,
        validation_data=(X_val_ft, y_val_ft),
        epochs=EPOCHS_ITER1,
        batch_size=BATCH_SIZE_ITER1,
        callbacks=get_callbacks(),
        verbose=0
    )

    hasil_iter1_per_plt[plt_name] = {
        "model": model_iter1,
        "history": history_iter1,
        "metode": "Transfer Learning"
    }

    print(f"Iterasi 1 selesai. Epoch berhenti pada epoch ke-{len(history_iter1.history['loss'])}")

In [ ]:
# ============================================================
# KEGIATAN 6: Iterasi 1 - Direct Training dengan Learning Rate Baru
# Berlaku untuk: PLTS Atap, PLT Hybrid (tetap TANPA bobot Pre-Training)
# ============================================================

for plt_name in PLT_DIRECT_TRAINING:
    print(f"\n=== Iterasi 1 (Direct Training) untuk: {plt_name} ===")

    X_train_ft = dataset_finetune_per_plt[plt_name]["X_train_ft"]
    y_train_ft = dataset_finetune_per_plt[plt_name]["y_train_ft"]
    X_val_ft = dataset_finetune_per_plt[plt_name]["X_val_ft"]
    y_val_ft = dataset_finetune_per_plt[plt_name]["y_val_ft"]

    # Arsitektur sama seperti Baseline, dilatih dari nol (tanpa bobot Pre-Training)
    model_iter1_direct = build_baseline_lstm(window_size=WINDOW_SIZE, n_features=N_FEATURES_REGIONAL)

    # Menggunakan learning rate Iterasi 1 juga, agar perbandingan tetap konsisten antar seluruh PLT
    model_iter1_direct.compile(optimizer=Adam(learning_rate=LR_ITER1), loss="mse", metrics=["mae"])

    history_iter1_direct = model_iter1_direct.fit(
        X_train_ft, y_train_ft,
        validation_data=(X_val_ft, y_val_ft),
        epochs=EPOCHS_ITER1,
        batch_size=BATCH_SIZE_ITER1,
        callbacks=get_callbacks(),
        verbose=0
    )

    hasil_iter1_per_plt[plt_name] = {
        "model": model_iter1_direct,
        "history": history_iter1_direct,
        "metode": "Direct Training"
    }

    print(f"Iterasi 1 selesai. Epoch berhenti pada epoch ke-{len(history_iter1_direct.history['loss'])}")

In [ ]:
# ============================================================
# KEGIATAN 7: Menampilkan Grafik Training Loss dan Validation Loss - Iterasi 1
# ============================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_final):
    history_iter1 = hasil_iter1_per_plt[plt_name]["history"]
    metode = hasil_iter1_per_plt[plt_name]["metode"]

    axes[i].plot(history_iter1.history["loss"], label="Training Loss")
    axes[i].plot(history_iter1.history["val_loss"], label="Validation Loss")
    axes[i].set_title(f"Loss Iterasi 1 - {plt_name} ({metode})")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Loss (MSE)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 8: Prediksi pada Data Uji 2025 dan Inverse Transform - Iterasi 1
# ============================================================

hasil_prediksi_iter1_per_plt = {}

for plt_name in daftar_plt_final:
    model_iter1 = hasil_iter1_per_plt[plt_name]["model"]
    scaler_target = scaler_target_regional_per_plt[plt_name]

    X_test = dataset_finetune_per_plt[plt_name]["X_test"]
    y_test = dataset_finetune_per_plt[plt_name]["y_test"]

    y_pred_scaled_iter1 = model_iter1.predict(X_test, verbose=0)

    y_pred_asli_iter1 = scaler_target.inverse_transform(y_pred_scaled_iter1)
    y_test_asli_iter1 = scaler_target.inverse_transform(y_test.reshape(-1, 1))

    hasil_prediksi_iter1_per_plt[plt_name] = {
        "y_test_asli": y_test_asli_iter1.flatten(),
        "y_pred_asli": y_pred_asli_iter1.flatten()
    }

    print(f"Prediksi Iterasi 1 untuk {plt_name} selesai. Jumlah data uji (2025): {len(y_test_asli_iter1)}")

In [ ]:
# ============================================================
# KEGIATAN 9: Evaluasi Model Menggunakan RMSE, MAE, dan MAPE - Iterasi 1
# ============================================================

evaluasi_iter1_per_plt = {}

for plt_name in daftar_plt_final:
    y_test_asli_iter1 = hasil_prediksi_iter1_per_plt[plt_name]["y_test_asli"]
    y_pred_asli_iter1 = hasil_prediksi_iter1_per_plt[plt_name]["y_pred_asli"]

    rmse_iter1 = np.sqrt(mean_squared_error(y_test_asli_iter1, y_pred_asli_iter1))
    mae_iter1 = mean_absolute_error(y_test_asli_iter1, y_pred_asli_iter1)
    mape_iter1 = hitung_mape(y_test_asli_iter1, y_pred_asli_iter1)

    evaluasi_iter1_per_plt[plt_name] = {"RMSE": rmse_iter1, "MAE": mae_iter1, "MAPE": mape_iter1}

    print(f"{plt_name:12s} -> RMSE: {rmse_iter1:8.3f} | MAE: {mae_iter1:8.3f} | MAPE: {mape_iter1:6.2f}%")

In [ ]:
# ============================================================
# KEGIATAN 10: Memvisualisasikan Perbandingan Aktual vs Prediksi - Iterasi 1
# ============================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_final):
    y_test_asli_iter1 = hasil_prediksi_iter1_per_plt[plt_name]["y_test_asli"]
    y_pred_asli_iter1 = hasil_prediksi_iter1_per_plt[plt_name]["y_pred_asli"]
    metode = hasil_iter1_per_plt[plt_name]["metode"]

    axes[i].plot(y_test_asli_iter1, label="Aktual", marker="o")
    axes[i].plot(y_pred_asli_iter1, label="Prediksi", marker="x")
    axes[i].set_title(f"Aktual vs Prediksi (Iterasi 1) - {plt_name} ({metode})")
    axes[i].set_xlabel("Periode (Data Uji 2025)")
    axes[i].set_ylabel("Produksi EBT (GWh)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 11: Menggabungkan Hasil Evaluasi Iterasi 1 ke DataFrame
# Diurutkan Berdasarkan RMSE
# ============================================================

hasil_iterasi1 = pd.DataFrame(evaluasi_iter1_per_plt).T
hasil_iterasi1.index.name = "Jenis_PLT"
hasil_iterasi1 = hasil_iterasi1.reset_index()

# Menambahkan kolom Metode (Transfer Learning / Direct Training)
hasil_iterasi1["Metode"] = hasil_iterasi1["Jenis_PLT"].apply(
    lambda plt_name: hasil_iter1_per_plt[plt_name]["metode"]
)

hasil_iterasi1 = hasil_iterasi1.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("Ringkasan Hasil Evaluasi Iterasi 1 (Learning Rate = 0.00005) per Jenis PLT:")
hasil_iterasi1

In [ ]:
# ============================================================
# [TASK GROUP 2] Evaluasi Iterasi 1 pada DATA VALIDASI (X_val_ft/y_val_ft,
# window sama dengan Fine-Tuning). Dipakai untuk PEMILIHAN model terbaik --
# lihat catatan yang sama di tahap Baseline di atas.
# ============================================================

evaluasi_iter1_val_per_plt = {}

for plt_name in daftar_plt_final:
    model_iter1 = hasil_iter1_per_plt[plt_name]["model"]
    scaler_target = scaler_target_regional_per_plt[plt_name]

    X_val_ft = dataset_finetune_per_plt[plt_name]["X_val_ft"]
    y_val_ft = dataset_finetune_per_plt[plt_name]["y_val_ft"]

    y_pred_scaled_val = model_iter1.predict(X_val_ft, verbose=0)
    y_pred_asli_val = scaler_target.inverse_transform(y_pred_scaled_val)
    y_val_asli = scaler_target.inverse_transform(y_val_ft.reshape(-1, 1))

    rmse_val = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli_val))
    mae_val = mean_absolute_error(y_val_asli, y_pred_asli_val)
    mape_val = hitung_mape(y_val_asli, y_pred_asli_val)

    evaluasi_iter1_val_per_plt[plt_name] = {"RMSE": rmse_val, "MAE": mae_val, "MAPE": mape_val}

    print(f"{plt_name:12s} -> [VALIDASI] RMSE: {rmse_val:8.3f} | MAE: {mae_val:8.3f} | MAPE: {mape_val:6.2f}%")

hasil_iterasi1_val = pd.DataFrame(evaluasi_iter1_val_per_plt).T
hasil_iterasi1_val.index.name = "Jenis_PLT"
hasil_iterasi1_val = hasil_iterasi1_val.reset_index()
hasil_iterasi1_val = hasil_iterasi1_val.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

In [ ]:
# ============================================================
# KEGIATAN 12: Tabel Perbandingan Fine-Tuning vs Iterasi 1
# (Untuk mengetahui apakah perubahan learning rate meningkatkan performa)
# ============================================================

df_perbandingan_iter1 = pd.merge(
    df_evaluasi_finetune[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
        columns={"RMSE": "RMSE_FineTuning", "MAE": "MAE_FineTuning", "MAPE": "MAPE_FineTuning"}
    ),
    hasil_iterasi1[["Jenis_PLT", "RMSE", "MAE", "MAPE", "Metode"]].rename(
        columns={"RMSE": "RMSE_Iterasi1", "MAE": "MAE_Iterasi1", "MAPE": "MAPE_Iterasi1"}
    ),
    on="Jenis_PLT",
    how="outer"
)

# Menghitung selisih dan persentase perubahan RMSE, MAE, MAPE akibat perubahan learning rate
df_perbandingan_iter1["Selisih_RMSE"] = (
    df_perbandingan_iter1["RMSE_FineTuning"] - df_perbandingan_iter1["RMSE_Iterasi1"]
)
df_perbandingan_iter1["Perubahan_RMSE (%)"] = (
    df_perbandingan_iter1["Selisih_RMSE"] / df_perbandingan_iter1["RMSE_FineTuning"] * 100
)
df_perbandingan_iter1["Selisih_MAE"] = (
    df_perbandingan_iter1["MAE_FineTuning"] - df_perbandingan_iter1["MAE_Iterasi1"]
)
df_perbandingan_iter1["Selisih_MAPE"] = (
    df_perbandingan_iter1["MAPE_FineTuning"] - df_perbandingan_iter1["MAPE_Iterasi1"]
)

# Keterangan: nilai positif pada kolom Selisih/Perubahan berarti Iterasi 1 LEBIH BAIK (error menurun)
df_perbandingan_iter1 = df_perbandingan_iter1.sort_values(
    by="RMSE_Iterasi1", ascending=True
).reset_index(drop=True)

print("Tabel Perbandingan Fine-Tuning vs Hyperparameter Tuning Iterasi 1:")
print("(Nilai positif pada kolom Selisih/Perubahan RMSE berarti Iterasi 1 lebih baik dari Fine-Tuning)")
df_perbandingan_iter1

In [ ]:
# ============================================================
# KEGIATAN 1: Menentukan Window Size Baru untuk Iterasi 2
# (Satu-satunya hyperparameter yang diubah pada iterasi ini)
# ============================================================

# Window size Fine-Tuning & Iterasi 1: 6 bulan
# Window size Iterasi 2: 12 bulan (data historis 1 tahun penuh sebagai input)
WINDOW_SIZE_ITER2 = 12

# Seluruh hyperparameter lain TETAP SAMA seperti Iterasi 1 (tidak diubah)
LR_ITER2 = LR_ITER1                # 0.00005, sama seperti Iterasi 1
EPOCHS_ITER2 = EPOCHS_ITER1
BATCH_SIZE_ITER2 = BATCH_SIZE_ITER1

print(f"Window size Iterasi 1 : {WINDOW_SIZE} bulan")
print(f"Window size Iterasi 2 : {WINDOW_SIZE_ITER2} bulan")
print(f"Learning rate (tetap) : {LR_ITER2}")

In [ ]:
# ============================================================
# KEGIATAN 2: Menggunakan Kembali Data Regional yang Sudah Dinormalisasi
# (data_scaled_regional_per_plt dari tahap Fine-Tuning, TIDAK preprocessing ulang)
# ============================================================

print("Data regional ternormalisasi (MinMaxScaler regional) digunakan kembali untuk Iterasi 2.")
print("Jenis PLT yang tersedia:", daftar_plt_final)

In [ ]:
# ============================================================
# KEGIATAN 3: Membentuk Sequence dengan Window Size = 12 Bulan
# (Fungsi create_sequences() TIDAK dibuat ulang, hanya parameter window yang disesuaikan)
# ============================================================

sequence_regional_iter2_per_plt = {}

for plt_name in daftar_plt_final:
    data_gabungan_scaled = data_scaled_regional_per_plt[plt_name]
    tanggal_plt = data_scaled_regional_per_plt[plt_name + "_tanggal"]

    # Menggunakan create_sequences() yang sama, hanya window_size diubah menjadi 12
    X_seq_iter2, y_seq_iter2 = create_sequences(data_gabungan_scaled, window_size=WINDOW_SIZE_ITER2)

    # Mengambil kolom target (Produksi) saja, yaitu kolom paling akhir
    y_target_seq_iter2 = y_seq_iter2[:, -1]

    # Tanggal disesuaikan dengan window baru (12 bulan)
    tanggal_target_iter2 = tanggal_plt[WINDOW_SIZE_ITER2:]

    sequence_regional_iter2_per_plt[plt_name] = {
        "X": X_seq_iter2,
        "y": y_target_seq_iter2,
        "tanggal_target": tanggal_target_iter2
    }

    print(f"{plt_name:12s} -> Total sequence (window=12) terbentuk: {len(X_seq_iter2)}")

In [ ]:
# ============================================================
# KEGIATAN 4: Split Data Berdasarkan Waktu (Tanpa Pengacakan) - Iterasi 2
# Training : 2023-2024   |   Test Akhir : 2025
# ============================================================

dataset_iter2_per_plt = {}

for plt_name in daftar_plt_final:
    X_seq_iter2 = sequence_regional_iter2_per_plt[plt_name]["X"]
    y_seq_iter2 = sequence_regional_iter2_per_plt[plt_name]["y"]
    tanggal_target_iter2 = pd.to_datetime(sequence_regional_iter2_per_plt[plt_name]["tanggal_target"])

    # Mask periode training (2023-2024) dan test akhir (2025) - sama seperti Fine-Tuning/Iterasi 1
    mask_train_iter2 = (tanggal_target_iter2.year >= 2023) & (tanggal_target_iter2.year <= 2024)
    mask_test_iter2 = tanggal_target_iter2.year == 2025

    X_train_full_iter2 = X_seq_iter2[mask_train_iter2]
    y_train_full_iter2 = y_seq_iter2[mask_train_iter2]
    X_test_iter2 = X_seq_iter2[mask_test_iter2]
    y_test_iter2 = y_seq_iter2[mask_test_iter2]

    # Split internal time-based untuk validasi (rasio sama seperti Fine-Tuning: VAL_INTERNAL_RATIO)
    split_idx_iter2 = int(len(X_train_full_iter2) * VAL_INTERNAL_RATIO)
    X_train_iter2 = X_train_full_iter2[:split_idx_iter2]
    y_train_iter2 = y_train_full_iter2[:split_idx_iter2]
    X_val_iter2 = X_train_full_iter2[split_idx_iter2:]
    y_val_iter2 = y_train_full_iter2[split_idx_iter2:]

    dataset_iter2_per_plt[plt_name] = {
        "X_train_iter2": X_train_iter2,
        "y_train_iter2": y_train_iter2,
        "X_val_iter2": X_val_iter2,
        "y_val_iter2": y_val_iter2,
        "X_test_iter2": X_test_iter2,
        "y_test_iter2": y_test_iter2,
    }

    print(f"{plt_name:12s} -> Train: {len(X_train_iter2):3d} | Val: {len(X_val_iter2):3d} | Test 2025: {len(X_test_iter2):3d}")

In [ ]:
# ============================================================
# KEGIATAN 5: Iterasi 2 - Transfer Learning dengan Window Size 12 Bulan
# Berlaku untuk: PLTA, PLTB, PLTM, PLTMH, PLTS
# (Bobot Pre-Training tetap dapat dimuat karena tidak bergantung pada window size)
# ============================================================

hasil_iter2_per_plt = {}

for plt_name in PLT_TRANSFER_LEARNING:
    print(f"\n=== Iterasi 2 (Transfer Learning, Window=12) untuk: {plt_name} ===")

    X_train_iter2 = dataset_iter2_per_plt[plt_name]["X_train_iter2"]
    y_train_iter2 = dataset_iter2_per_plt[plt_name]["y_train_iter2"]
    X_val_iter2 = dataset_iter2_per_plt[plt_name]["X_val_iter2"]
    y_val_iter2 = dataset_iter2_per_plt[plt_name]["y_val_iter2"]

    # 1. Membangun arsitektur yang SAMA seperti Baseline, hanya window_size yang berbeda (12)
    model_iter2 = build_baseline_lstm(window_size=WINDOW_SIZE_ITER2, n_features=N_FEATURES_REGIONAL)

    # 2. Memuat bobot hasil Pre-Training sebagai bobot awal
    #    (bobot LSTM tidak bergantung pada panjang window, sehingga tetap kompatibel)
    bobot_pretrain = hasil_pretrain_per_plt[plt_name]["model"].get_weights()
    model_iter2.set_weights(bobot_pretrain)

    # 3. Compile dengan learning rate yang SAMA seperti Iterasi 1 (LR_ITER2 = LR_ITER1)
    model_iter2.compile(optimizer=Adam(learning_rate=LR_ITER2), loss="mse", metrics=["mae"])

    # 4. Training dengan epoch, batch size, dan callback yang sama seperti Iterasi 1
    history_iter2 = model_iter2.fit(
        X_train_iter2, y_train_iter2,
        validation_data=(X_val_iter2, y_val_iter2),
        epochs=EPOCHS_ITER2,
        batch_size=BATCH_SIZE_ITER2,
        callbacks=get_callbacks(),
        verbose=0
    )

    hasil_iter2_per_plt[plt_name] = {
        "model": model_iter2,
        "history": history_iter2,
        "metode": "Transfer Learning"
    }

    print(f"Iterasi 2 selesai. Epoch berhenti pada epoch ke-{len(history_iter2.history['loss'])}")

In [ ]:
# ============================================================
# KEGIATAN 6: Iterasi 2 - Direct Training dengan Window Size 12 Bulan
# Berlaku untuk: PLTS Atap, PLT Hybrid (tetap TANPA bobot Pre-Training)
# ============================================================

for plt_name in PLT_DIRECT_TRAINING:
    print(f"\n=== Iterasi 2 (Direct Training, Window=12) untuk: {plt_name} ===")

    X_train_iter2 = dataset_iter2_per_plt[plt_name]["X_train_iter2"]
    y_train_iter2 = dataset_iter2_per_plt[plt_name]["y_train_iter2"]
    X_val_iter2 = dataset_iter2_per_plt[plt_name]["X_val_iter2"]
    y_val_iter2 = dataset_iter2_per_plt[plt_name]["y_val_iter2"]

    # Arsitektur sama, dilatih dari nol dengan window_size=12
    model_iter2_direct = build_baseline_lstm(window_size=WINDOW_SIZE_ITER2, n_features=N_FEATURES_REGIONAL)
    model_iter2_direct.compile(optimizer=Adam(learning_rate=LR_ITER2), loss="mse", metrics=["mae"])

    history_iter2_direct = model_iter2_direct.fit(
        X_train_iter2, y_train_iter2,
        validation_data=(X_val_iter2, y_val_iter2),
        epochs=EPOCHS_ITER2,
        batch_size=BATCH_SIZE_ITER2,
        callbacks=get_callbacks(),
        verbose=0
    )

    hasil_iter2_per_plt[plt_name] = {
        "model": model_iter2_direct,
        "history": history_iter2_direct,
        "metode": "Direct Training"
    }

    print(f"Iterasi 2 selesai. Epoch berhenti pada epoch ke-{len(history_iter2_direct.history['loss'])}")

In [ ]:
# ============================================================
# KEGIATAN 7: Menampilkan Grafik Training Loss dan Validation Loss - Iterasi 2
# ============================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_final):
    history_iter2 = hasil_iter2_per_plt[plt_name]["history"]
    metode = hasil_iter2_per_plt[plt_name]["metode"]

    axes[i].plot(history_iter2.history["loss"], label="Training Loss")
    axes[i].plot(history_iter2.history["val_loss"], label="Validation Loss")
    axes[i].set_title(f"Loss Iterasi 2 (Window=12) - {plt_name} ({metode})")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Loss (MSE)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 8: Prediksi pada Data Uji 2025 dan Inverse Transform - Iterasi 2
# ============================================================

hasil_prediksi_iter2_per_plt = {}

for plt_name in daftar_plt_final:
    model_iter2 = hasil_iter2_per_plt[plt_name]["model"]
    scaler_target = scaler_target_regional_per_plt[plt_name]

    X_test_iter2 = dataset_iter2_per_plt[plt_name]["X_test_iter2"]
    y_test_iter2 = dataset_iter2_per_plt[plt_name]["y_test_iter2"]

    y_pred_scaled_iter2 = model_iter2.predict(X_test_iter2, verbose=0)

    y_pred_asli_iter2 = scaler_target.inverse_transform(y_pred_scaled_iter2)
    y_test_asli_iter2 = scaler_target.inverse_transform(y_test_iter2.reshape(-1, 1))

    hasil_prediksi_iter2_per_plt[plt_name] = {
        "y_test_asli": y_test_asli_iter2.flatten(),
        "y_pred_asli": y_pred_asli_iter2.flatten()
    }

    print(f"Prediksi Iterasi 2 untuk {plt_name} selesai. Jumlah data uji (2025): {len(y_test_asli_iter2)}")

In [ ]:
# ============================================================
# KEGIATAN 9: Evaluasi Model Menggunakan RMSE, MAE, dan MAPE - Iterasi 2
# ============================================================

evaluasi_iter2_per_plt = {}

for plt_name in daftar_plt_final:
    y_test_asli_iter2 = hasil_prediksi_iter2_per_plt[plt_name]["y_test_asli"]
    y_pred_asli_iter2 = hasil_prediksi_iter2_per_plt[plt_name]["y_pred_asli"]

    rmse_iter2 = np.sqrt(mean_squared_error(y_test_asli_iter2, y_pred_asli_iter2))
    mae_iter2 = mean_absolute_error(y_test_asli_iter2, y_pred_asli_iter2)
    mape_iter2 = hitung_mape(y_test_asli_iter2, y_pred_asli_iter2)

    evaluasi_iter2_per_plt[plt_name] = {"RMSE": rmse_iter2, "MAE": mae_iter2, "MAPE": mape_iter2}

    print(f"{plt_name:12s} -> RMSE: {rmse_iter2:8.3f} | MAE: {mae_iter2:8.3f} | MAPE: {mape_iter2:6.2f}%")

In [ ]:
# ============================================================
# KEGIATAN 10: Memvisualisasikan Perbandingan Aktual vs Prediksi - Iterasi 2
# ============================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_final):
    y_test_asli_iter2 = hasil_prediksi_iter2_per_plt[plt_name]["y_test_asli"]
    y_pred_asli_iter2 = hasil_prediksi_iter2_per_plt[plt_name]["y_pred_asli"]
    metode = hasil_iter2_per_plt[plt_name]["metode"]

    axes[i].plot(y_test_asli_iter2, label="Aktual", marker="o")
    axes[i].plot(y_pred_asli_iter2, label="Prediksi", marker="x")
    axes[i].set_title(f"Aktual vs Prediksi (Iterasi 2, Window=12) - {plt_name} ({metode})")
    axes[i].set_xlabel("Periode (Data Uji 2025)")
    axes[i].set_ylabel("Produksi EBT (GWh)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 11: Menggabungkan Hasil Evaluasi Iterasi 2 ke DataFrame
# Diurutkan Berdasarkan RMSE
# ============================================================

hasil_iterasi2 = pd.DataFrame(evaluasi_iter2_per_plt).T
hasil_iterasi2.index.name = "Jenis_PLT"
hasil_iterasi2 = hasil_iterasi2.reset_index()

hasil_iterasi2["Metode"] = hasil_iterasi2["Jenis_PLT"].apply(
    lambda plt_name: hasil_iter2_per_plt[plt_name]["metode"]
)

hasil_iterasi2 = hasil_iterasi2.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("Ringkasan Hasil Evaluasi Iterasi 2 (Window Size = 12 Bulan) per Jenis PLT:")
hasil_iterasi2

In [ ]:
# ============================================================
# [TASK GROUP 2] Evaluasi Iterasi 2 pada DATA VALIDASI
# (X_val_iter2/y_val_iter2, window=12 -- BUKAN X_val_ft, karena Iterasi 2
# pakai sequence window berbeda). Dipakai untuk PEMILIHAN model terbaik --
# lihat catatan yang sama di tahap Baseline di atas.
# ============================================================

evaluasi_iter2_val_per_plt = {}

for plt_name in daftar_plt_final:
    model_iter2 = hasil_iter2_per_plt[plt_name]["model"]
    scaler_target = scaler_target_regional_per_plt[plt_name]

    X_val_iter2 = dataset_iter2_per_plt[plt_name]["X_val_iter2"]
    y_val_iter2 = dataset_iter2_per_plt[plt_name]["y_val_iter2"]

    y_pred_scaled_val = model_iter2.predict(X_val_iter2, verbose=0)
    y_pred_asli_val = scaler_target.inverse_transform(y_pred_scaled_val)
    y_val_asli = scaler_target.inverse_transform(y_val_iter2.reshape(-1, 1))

    rmse_val = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli_val))
    mae_val = mean_absolute_error(y_val_asli, y_pred_asli_val)
    mape_val = hitung_mape(y_val_asli, y_pred_asli_val)

    evaluasi_iter2_val_per_plt[plt_name] = {"RMSE": rmse_val, "MAE": mae_val, "MAPE": mape_val}

    print(f"{plt_name:12s} -> [VALIDASI] RMSE: {rmse_val:8.3f} | MAE: {mae_val:8.3f} | MAPE: {mape_val:6.2f}%")

hasil_iterasi2_val = pd.DataFrame(evaluasi_iter2_val_per_plt).T
hasil_iterasi2_val.index.name = "Jenis_PLT"
hasil_iterasi2_val = hasil_iterasi2_val.reset_index()
hasil_iterasi2_val = hasil_iterasi2_val.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

In [ ]:
# ============================================================
# KEGIATAN 1: Menentukan Dropout Rate Baru untuk Iterasi 3
# (Satu-satunya hyperparameter yang diubah pada iterasi ini)
# ============================================================

# Dropout Fine-Tuning & Iterasi 1 & Iterasi 2: 0.20 (default build_baseline_lstm)
# Dropout Iterasi 3: 0.30
DROPOUT_ITER3 = 0.30

# Konfigurasi terbaik dari Iterasi 1 digunakan kembali (window=6, learning rate=0.00005)
WINDOW_SIZE_ITER3 = WINDOW_SIZE       # 6 bulan (sama seperti Fine-Tuning/Iterasi 1)
LR_ITER3 = LR_ITER1                   # 0.00005 (sama seperti Iterasi 1)
EPOCHS_ITER3 = EPOCHS_ITER1
BATCH_SIZE_ITER3 = BATCH_SIZE_ITER1

print(f"Window size Iterasi 3 : {WINDOW_SIZE_ITER3} bulan (mengikuti Iterasi 1)")
print(f"Learning rate Iterasi 3 : {LR_ITER3} (mengikuti Iterasi 1)")
print(f"Dropout Iterasi 3 : {DROPOUT_ITER3} (naik dari 0.20)")

In [ ]:
# ============================================================
# KEGIATAN 2: Menggunakan Kembali Dataset dengan Window Size 6 Bulan
# (dataset_finetune_per_plt, TIDAK dibentuk ulang karena window kembali ke 6)
# ============================================================

print("Dataset regional (window=6, hasil split Fine-Tuning/Iterasi 1) digunakan kembali untuk Iterasi 3.")
for plt_name in daftar_plt_final:
    n_train = len(dataset_finetune_per_plt[plt_name]["X_train_ft"])
    n_val = len(dataset_finetune_per_plt[plt_name]["X_val_ft"])
    n_test = len(dataset_finetune_per_plt[plt_name]["X_test"])
    print(f"{plt_name:12s} -> Train: {n_train:3d} | Val: {n_val:3d} | Test 2025: {n_test:3d}")

In [ ]:
# ============================================================
# KEGIATAN 3: Iterasi 3 - Transfer Learning dengan Dropout 0.30
# Berlaku untuk: PLTA, PLTB, PLTM, PLTMH, PLTS
# ============================================================

hasil_iter3_per_plt = {}

for plt_name in PLT_TRANSFER_LEARNING:
    print(f"\n=== Iterasi 3 (Transfer Learning, Dropout=0.30) untuk: {plt_name} ===")

    X_train_ft = dataset_finetune_per_plt[plt_name]["X_train_ft"]
    y_train_ft = dataset_finetune_per_plt[plt_name]["y_train_ft"]
    X_val_ft = dataset_finetune_per_plt[plt_name]["X_val_ft"]
    y_val_ft = dataset_finetune_per_plt[plt_name]["y_val_ft"]

    # 1. Membangun arsitektur SAMA seperti Baseline, hanya dropout_rate yang diubah (0.30)
    model_iter3 = build_baseline_lstm(
        window_size=WINDOW_SIZE_ITER3,
        n_features=N_FEATURES_REGIONAL,
        dropout_rate=DROPOUT_ITER3
    )

    # 2. Memuat bobot hasil Pre-Training sebagai bobot awal
    #    (dropout tidak memiliki bobot, sehingga tidak memengaruhi kompatibilitas set_weights)
    bobot_pretrain = hasil_pretrain_per_plt[plt_name]["model"].get_weights()
    model_iter3.set_weights(bobot_pretrain)

    # 3. Compile dengan learning rate SAMA seperti Iterasi 1
    model_iter3.compile(optimizer=Adam(learning_rate=LR_ITER3), loss="mse", metrics=["mae"])

    # 4. Training dengan epoch, batch size, callback yang sama seperti Iterasi 1
    history_iter3 = model_iter3.fit(
        X_train_ft, y_train_ft,
        validation_data=(X_val_ft, y_val_ft),
        epochs=EPOCHS_ITER3,
        batch_size=BATCH_SIZE_ITER3,
        callbacks=get_callbacks(),
        verbose=0
    )

    hasil_iter3_per_plt[plt_name] = {
        "model": model_iter3,
        "history": history_iter3,
        "metode": "Transfer Learning"
    }

    print(f"Iterasi 3 selesai. Epoch berhenti pada epoch ke-{len(history_iter3.history['loss'])}")

In [ ]:
# ============================================================
# KEGIATAN 4: Iterasi 3 - Direct Training dengan Dropout 0.30
# Berlaku untuk: PLTS Atap, PLT Hybrid (tetap TANPA bobot Pre-Training)
# ============================================================

for plt_name in PLT_DIRECT_TRAINING:
    print(f"\n=== Iterasi 3 (Direct Training, Dropout=0.30) untuk: {plt_name} ===")

    X_train_ft = dataset_finetune_per_plt[plt_name]["X_train_ft"]
    y_train_ft = dataset_finetune_per_plt[plt_name]["y_train_ft"]
    X_val_ft = dataset_finetune_per_plt[plt_name]["X_val_ft"]
    y_val_ft = dataset_finetune_per_plt[plt_name]["y_val_ft"]

    model_iter3_direct = build_baseline_lstm(
        window_size=WINDOW_SIZE_ITER3,
        n_features=N_FEATURES_REGIONAL,
        dropout_rate=DROPOUT_ITER3
    )
    model_iter3_direct.compile(optimizer=Adam(learning_rate=LR_ITER3), loss="mse", metrics=["mae"])

    history_iter3_direct = model_iter3_direct.fit(
        X_train_ft, y_train_ft,
        validation_data=(X_val_ft, y_val_ft),
        epochs=EPOCHS_ITER3,
        batch_size=BATCH_SIZE_ITER3,
        callbacks=get_callbacks(),
        verbose=0
    )

    hasil_iter3_per_plt[plt_name] = {
        "model": model_iter3_direct,
        "history": history_iter3_direct,
        "metode": "Direct Training"
    }

    print(f"Iterasi 3 selesai. Epoch berhenti pada epoch ke-{len(history_iter3_direct.history['loss'])}")

In [ ]:
# ============================================================
# KEGIATAN 5: Menampilkan Grafik Training Loss dan Validation Loss - Iterasi 3
# ============================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_final):
    history_iter3 = hasil_iter3_per_plt[plt_name]["history"]
    metode = hasil_iter3_per_plt[plt_name]["metode"]

    axes[i].plot(history_iter3.history["loss"], label="Training Loss")
    axes[i].plot(history_iter3.history["val_loss"], label="Validation Loss")
    axes[i].set_title(f"Loss Iterasi 3 (Dropout=0.30) - {plt_name} ({metode})")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Loss (MSE)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 6: Prediksi pada Data Uji 2025 dan Inverse Transform - Iterasi 3
# ============================================================

hasil_prediksi_iter3_per_plt = {}

for plt_name in daftar_plt_final:
    model_iter3 = hasil_iter3_per_plt[plt_name]["model"]
    scaler_target = scaler_target_regional_per_plt[plt_name]

    X_test = dataset_finetune_per_plt[plt_name]["X_test"]
    y_test = dataset_finetune_per_plt[plt_name]["y_test"]

    y_pred_scaled_iter3 = model_iter3.predict(X_test, verbose=0)

    y_pred_asli_iter3 = scaler_target.inverse_transform(y_pred_scaled_iter3)
    y_test_asli_iter3 = scaler_target.inverse_transform(y_test.reshape(-1, 1))

    hasil_prediksi_iter3_per_plt[plt_name] = {
        "y_test_asli": y_test_asli_iter3.flatten(),
        "y_pred_asli": y_pred_asli_iter3.flatten()
    }

    print(f"Prediksi Iterasi 3 untuk {plt_name} selesai. Jumlah data uji (2025): {len(y_test_asli_iter3)}")

In [ ]:
# ============================================================
# KEGIATAN 7: Evaluasi Model Menggunakan RMSE, MAE, dan MAPE - Iterasi 3
# ============================================================

evaluasi_iter3_per_plt = {}

for plt_name in daftar_plt_final:
    y_test_asli_iter3 = hasil_prediksi_iter3_per_plt[plt_name]["y_test_asli"]
    y_pred_asli_iter3 = hasil_prediksi_iter3_per_plt[plt_name]["y_pred_asli"]

    rmse_iter3 = np.sqrt(mean_squared_error(y_test_asli_iter3, y_pred_asli_iter3))
    mae_iter3 = mean_absolute_error(y_test_asli_iter3, y_pred_asli_iter3)
    mape_iter3 = hitung_mape(y_test_asli_iter3, y_pred_asli_iter3)

    evaluasi_iter3_per_plt[plt_name] = {"RMSE": rmse_iter3, "MAE": mae_iter3, "MAPE": mape_iter3}

    print(f"{plt_name:12s} -> RMSE: {rmse_iter3:8.3f} | MAE: {mae_iter3:8.3f} | MAPE: {mape_iter3:6.2f}%")

In [ ]:
# ============================================================
# KEGIATAN 8: Memvisualisasikan Perbandingan Aktual vs Prediksi - Iterasi 3
# ============================================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(daftar_plt_final):
    y_test_asli_iter3 = hasil_prediksi_iter3_per_plt[plt_name]["y_test_asli"]
    y_pred_asli_iter3 = hasil_prediksi_iter3_per_plt[plt_name]["y_pred_asli"]
    metode = hasil_iter3_per_plt[plt_name]["metode"]

    axes[i].plot(y_test_asli_iter3, label="Aktual", marker="o")
    axes[i].plot(y_pred_asli_iter3, label="Prediksi", marker="x")
    axes[i].set_title(f"Aktual vs Prediksi (Iterasi 3, Dropout=0.30) - {plt_name} ({metode})")
    axes[i].set_xlabel("Periode (Data Uji 2025)")
    axes[i].set_ylabel("Produksi EBT (GWh)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 9: Menggabungkan Hasil Evaluasi Iterasi 3 ke DataFrame
# Diurutkan Berdasarkan RMSE
# ============================================================

hasil_iterasi3 = pd.DataFrame(evaluasi_iter3_per_plt).T
hasil_iterasi3.index.name = "Jenis_PLT"
hasil_iterasi3 = hasil_iterasi3.reset_index()

hasil_iterasi3["Metode"] = hasil_iterasi3["Jenis_PLT"].apply(
    lambda plt_name: hasil_iter3_per_plt[plt_name]["metode"]
)

hasil_iterasi3 = hasil_iterasi3.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("Ringkasan Hasil Evaluasi Iterasi 3 (Dropout = 0.30) per Jenis PLT:")
hasil_iterasi3

In [ ]:
# ============================================================
# KEGIATAN 7: Evaluasi Model Menggunakan RMSE, MAE, dan MAPE - Iterasi 3
# ============================================================

evaluasi_iter3_per_plt = {}

for plt_name in daftar_plt_final:
    y_test_asli_iter3 = hasil_prediksi_iter3_per_plt[plt_name]["y_test_asli"]
    y_pred_asli_iter3 = hasil_prediksi_iter3_per_plt[plt_name]["y_pred_asli"]

    rmse_iter3 = np.sqrt(mean_squared_error(y_test_asli_iter3, y_pred_asli_iter3))
    mae_iter3 = mean_absolute_error(y_test_asli_iter3, y_pred_asli_iter3)
    mape_iter3 = hitung_mape(y_test_asli_iter3, y_pred_asli_iter3)

    evaluasi_iter3_per_plt[plt_name] = {"RMSE": rmse_iter3, "MAE": mae_iter3, "MAPE": mape_iter3}

    print(f"{plt_name:12s} -> RMSE: {rmse_iter3:8.3f} | MAE: {mae_iter3:8.3f} | MAPE: {mape_iter3:6.2f}%")

In [ ]:
# ============================================================
# KEGIATAN 9: Menggabungkan Hasil Evaluasi Iterasi 3 ke DataFrame
# Diurutkan Berdasarkan RMSE
# ============================================================

hasil_iterasi3 = pd.DataFrame(evaluasi_iter3_per_plt).T
hasil_iterasi3.index.name = "Jenis_PLT"
hasil_iterasi3 = hasil_iterasi3.reset_index()

hasil_iterasi3["Metode"] = hasil_iterasi3["Jenis_PLT"].apply(
    lambda plt_name: hasil_iter3_per_plt[plt_name]["metode"]
)

hasil_iterasi3 = hasil_iterasi3.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("Ringkasan Hasil Evaluasi Iterasi 3 (Dropout = 0.30) per Jenis PLT:")
hasil_iterasi3

In [ ]:
# ============================================================
# [TASK GROUP 2] Evaluasi Iterasi 3 pada DATA VALIDASI (X_val_ft/y_val_ft,
# window sama dengan Fine-Tuning/Iterasi 1). Dipakai untuk PEMILIHAN model
# terbaik -- lihat catatan yang sama di tahap Baseline di atas.
# ============================================================

evaluasi_iter3_val_per_plt = {}

for plt_name in daftar_plt_final:
    model_iter3 = hasil_iter3_per_plt[plt_name]["model"]
    scaler_target = scaler_target_regional_per_plt[plt_name]

    X_val_ft = dataset_finetune_per_plt[plt_name]["X_val_ft"]
    y_val_ft = dataset_finetune_per_plt[plt_name]["y_val_ft"]

    y_pred_scaled_val = model_iter3.predict(X_val_ft, verbose=0)
    y_pred_asli_val = scaler_target.inverse_transform(y_pred_scaled_val)
    y_val_asli = scaler_target.inverse_transform(y_val_ft.reshape(-1, 1))

    rmse_val = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli_val))
    mae_val = mean_absolute_error(y_val_asli, y_pred_asli_val)
    mape_val = hitung_mape(y_val_asli, y_pred_asli_val)

    evaluasi_iter3_val_per_plt[plt_name] = {"RMSE": rmse_val, "MAE": mae_val, "MAPE": mape_val}

    print(f"{plt_name:12s} -> [VALIDASI] RMSE: {rmse_val:8.3f} | MAE: {mae_val:8.3f} | MAPE: {mape_val:6.2f}%")

hasil_iterasi3_val = pd.DataFrame(evaluasi_iter3_val_per_plt).T
hasil_iterasi3_val.index.name = "Jenis_PLT"
hasil_iterasi3_val = hasil_iterasi3_val.reset_index()
hasil_iterasi3_val = hasil_iterasi3_val.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

In [ ]:
# ============================================================
# KEGIATAN 10: Tabel Perbandingan Lengkap Seluruh Tahap
# (Baseline, Fine-Tuning, Iterasi 1, Iterasi 2, Iterasi 3)
# + Penentuan Model Terbaik Berdasarkan RMSE dan MAPE Terendah
# ============================================================

# Menyiapkan masing-masing tabel evaluasi dengan penamaan kolom yang jelas per tahap
tabel_baseline = df_evaluasi[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Baseline", "MAE": "MAE_Baseline", "MAPE": "MAPE_Baseline"}
)
tabel_finetuning = df_evaluasi_finetune[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_FineTuning", "MAE": "MAE_FineTuning", "MAPE": "MAPE_FineTuning"}
)
tabel_iterasi1 = hasil_iterasi1[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Iterasi1", "MAE": "MAE_Iterasi1", "MAPE": "MAPE_Iterasi1"}
)
tabel_iterasi2 = hasil_iterasi2[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Iterasi2", "MAE": "MAE_Iterasi2", "MAPE": "MAPE_Iterasi2"}
)
tabel_iterasi3 = hasil_iterasi3[["Jenis_PLT", "RMSE", "MAE", "MAPE", "Metode"]].rename(
    columns={"RMSE": "RMSE_Iterasi3", "MAE": "MAE_Iterasi3", "MAPE": "MAPE_Iterasi3"}
)

# Menggabungkan seluruh tabel berdasarkan Jenis_PLT (outer join agar PLTS Atap & PLT Hybrid tetap tampil
# meskipun tidak memiliki hasil Baseline)
df_perbandingan_lengkap = tabel_baseline.merge(tabel_finetuning, on="Jenis_PLT", how="outer") \
                                         .merge(tabel_iterasi1, on="Jenis_PLT", how="outer") \
                                         .merge(tabel_iterasi2, on="Jenis_PLT", how="outer") \
                                         .merge(tabel_iterasi3, on="Jenis_PLT", how="outer")

# Menentukan tahap dengan RMSE terendah untuk setiap jenis PLT
kolom_rmse = ["RMSE_Baseline", "RMSE_FineTuning", "RMSE_Iterasi1", "RMSE_Iterasi2", "RMSE_Iterasi3"]
peta_nama_tahap_rmse = {
    "RMSE_Baseline": "Baseline", "RMSE_FineTuning": "Fine-Tuning",
    "RMSE_Iterasi1": "Iterasi 1", "RMSE_Iterasi2": "Iterasi 2", "RMSE_Iterasi3": "Iterasi 3"
}
df_perbandingan_lengkap["Model_Terbaik_RMSE"] = df_perbandingan_lengkap[kolom_rmse].idxmin(axis=1).map(
    peta_nama_tahap_rmse
)

# Menentukan tahap dengan MAPE terendah untuk setiap jenis PLT
kolom_mape = ["MAPE_Baseline", "MAPE_FineTuning", "MAPE_Iterasi1", "MAPE_Iterasi2", "MAPE_Iterasi3"]
peta_nama_tahap_mape = {
    "MAPE_Baseline": "Baseline", "MAPE_FineTuning": "Fine-Tuning",
    "MAPE_Iterasi1": "Iterasi 1", "MAPE_Iterasi2": "Iterasi 2", "MAPE_Iterasi3": "Iterasi 3"
}
df_perbandingan_lengkap["Model_Terbaik_MAPE"] = df_perbandingan_lengkap[kolom_mape].idxmin(axis=1).map(
    peta_nama_tahap_mape
)

# Mengurutkan berdasarkan RMSE Iterasi 3 (tahap akhir) sebagai acuan utama
df_perbandingan_lengkap = df_perbandingan_lengkap.sort_values(
    by="RMSE_Iterasi3", ascending=True
).reset_index(drop=True)

print("Tabel Perbandingan Lengkap: Baseline vs Fine-Tuning vs Iterasi 1 vs Iterasi 2 vs Iterasi 3")
print("(PLTS Atap & PLT Hybrid tidak memiliki nilai Baseline karena tidak tersedia pada data nasional)")
df_perbandingan_lengkap

In [ ]:
# ============================================================
# KEGIATAN 1: Menampilkan Kembali Seluruh DataFrame Hasil Evaluasi
# (Untuk verifikasi sebelum digabungkan)
# ============================================================

print("1. Hasil Evaluasi Baseline LSTM:")
display(df_evaluasi)

print("\n2. Hasil Evaluasi Fine-Tuning (Transfer Learning):")
display(df_evaluasi_finetune)

print("\n3. Hasil Evaluasi Iterasi 1 (Learning Rate = 0.00005):")
display(hasil_iterasi1)

print("\n4. Hasil Evaluasi Iterasi 2 (Window Size = 12):")
display(hasil_iterasi2)

print("\n5. Hasil Evaluasi Iterasi 3 (Dropout = 0.30):")
display(hasil_iterasi3)

In [ ]:
# ============================================================
# KEGIATAN 2: Menggabungkan Seluruh Hasil Evaluasi ke Satu DataFrame
# Berdasarkan Jenis_PLT (RMSE, MAE, MAPE dari setiap model)
# ============================================================

tabel_baseline_pm = df_evaluasi[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Baseline", "MAE": "MAE_Baseline", "MAPE": "MAPE_Baseline"}
)
tabel_finetuning_pm = df_evaluasi_finetune[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_FineTuning", "MAE": "MAE_FineTuning", "MAPE": "MAPE_FineTuning"}
)
tabel_iterasi1_pm = hasil_iterasi1[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Iterasi1", "MAE": "MAE_Iterasi1", "MAPE": "MAPE_Iterasi1"}
)
tabel_iterasi2_pm = hasil_iterasi2[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Iterasi2", "MAE": "MAE_Iterasi2", "MAPE": "MAPE_Iterasi2"}
)
tabel_iterasi3_pm = hasil_iterasi3[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Iterasi3", "MAE": "MAE_Iterasi3", "MAPE": "MAPE_Iterasi3"}
)

# Menggabungkan seluruh tabel berdasarkan Jenis_PLT (outer join agar PLTS Atap & PLT Hybrid tetap tampil)
perbandingan_model = tabel_baseline_pm.merge(tabel_finetuning_pm, on="Jenis_PLT", how="outer") \
                                       .merge(tabel_iterasi1_pm, on="Jenis_PLT", how="outer") \
                                       .merge(tabel_iterasi2_pm, on="Jenis_PLT", how="outer") \
                                       .merge(tabel_iterasi3_pm, on="Jenis_PLT", how="outer")

print("DataFrame Perbandingan Seluruh Model (RMSE, MAE, MAPE) per Jenis PLT:")
perbandingan_model

In [ ]:
# ============================================================
# [TASK GROUP 2] Tabel Perbandingan Seluruh Model BERDASARKAN DATA VALIDASI
# (bukan test 2025) -- ini yang dipakai untuk PEMILIHAN model terbaik di
# KEGIATAN 4/5 di bawah, supaya data test 2025 tidak ikut menentukan model
# mana yang menang (test-set leakage pada seleksi model).
# ============================================================

tabel_baseline_val_pm = df_evaluasi_val[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Baseline_Val", "MAE": "MAE_Baseline_Val", "MAPE": "MAPE_Baseline_Val"}
)
tabel_finetuning_val_pm = df_evaluasi_finetune_val[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_FineTuning_Val", "MAE": "MAE_FineTuning_Val", "MAPE": "MAPE_FineTuning_Val"}
)
tabel_iterasi1_val_pm = hasil_iterasi1_val[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Iterasi1_Val", "MAE": "MAE_Iterasi1_Val", "MAPE": "MAPE_Iterasi1_Val"}
)
tabel_iterasi2_val_pm = hasil_iterasi2_val[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Iterasi2_Val", "MAE": "MAE_Iterasi2_Val", "MAPE": "MAPE_Iterasi2_Val"}
)
tabel_iterasi3_val_pm = hasil_iterasi3_val[["Jenis_PLT", "RMSE", "MAE", "MAPE"]].rename(
    columns={"RMSE": "RMSE_Iterasi3_Val", "MAE": "MAE_Iterasi3_Val", "MAPE": "MAPE_Iterasi3_Val"}
)

perbandingan_model_VALIDASI = tabel_baseline_val_pm.merge(tabel_finetuning_val_pm, on="Jenis_PLT", how="outer") \
                                                     .merge(tabel_iterasi1_val_pm, on="Jenis_PLT", how="outer") \
                                                     .merge(tabel_iterasi2_val_pm, on="Jenis_PLT", how="outer") \
                                                     .merge(tabel_iterasi3_val_pm, on="Jenis_PLT", how="outer")

print("DataFrame Perbandingan Seluruh Model BERDASARKAN VALIDASI (RMSE, MAE, MAPE) per Jenis PLT:")
perbandingan_model_VALIDASI

In [ ]:
# ============================================================
# KEGIATAN 3: Menghitung Rata-Rata RMSE, MAE, MAPE untuk Setiap Model
# (Performa keseluruhan, bukan hanya per jenis PLT)
# ============================================================

daftar_nama_model = ["Baseline", "FineTuning", "Iterasi1", "Iterasi2", "Iterasi3"]

ringkasan_model = pd.DataFrame({
    "Model": daftar_nama_model,
    "Average_RMSE": [
        perbandingan_model["RMSE_Baseline"].mean(),
        perbandingan_model["RMSE_FineTuning"].mean(),
        perbandingan_model["RMSE_Iterasi1"].mean(),
        perbandingan_model["RMSE_Iterasi2"].mean(),
        perbandingan_model["RMSE_Iterasi3"].mean(),
    ],
    "Average_MAE": [
        perbandingan_model["MAE_Baseline"].mean(),
        perbandingan_model["MAE_FineTuning"].mean(),
        perbandingan_model["MAE_Iterasi1"].mean(),
        perbandingan_model["MAE_Iterasi2"].mean(),
        perbandingan_model["MAE_Iterasi3"].mean(),
    ],
    "Average_MAPE": [
        perbandingan_model["MAPE_Baseline"].mean(),
        perbandingan_model["MAPE_FineTuning"].mean(),
        perbandingan_model["MAPE_Iterasi1"].mean(),
        perbandingan_model["MAPE_Iterasi2"].mean(),
        perbandingan_model["MAPE_Iterasi3"].mean(),
    ],
})

# Catatan: rata-rata Baseline dihitung hanya dari PLT yang memiliki data nasional
# (PLTS Atap & PLT Hybrid bernilai NaN pada kolom Baseline, otomatis diabaikan oleh .mean())

ringkasan_model = ringkasan_model.sort_values(by="Average_RMSE", ascending=True).reset_index(drop=True)

print("Ringkasan Rata-Rata Performa Seluruh Model (TEST 2025 -- untuk pelaporan/konfirmasi akhir):")
ringkasan_model

In [ ]:
# ============================================================
# [TASK GROUP 2] Ringkasan Rata-Rata Performa BERDASARKAN VALIDASI
# Ini yang dipakai untuk KEGIATAN 4 (penentuan model_terbaik) di bawah --
# ringkasan_model (test 2025) di atas TETAP dihitung untuk pelaporan/
# konfirmasi akhir, bukan lagi dasar pemilihan.
# ============================================================

ringkasan_model_validasi = pd.DataFrame({
    "Model": daftar_nama_model,
    "Average_RMSE": [
        perbandingan_model_VALIDASI["RMSE_Baseline_Val"].mean(),
        perbandingan_model_VALIDASI["RMSE_FineTuning_Val"].mean(),
        perbandingan_model_VALIDASI["RMSE_Iterasi1_Val"].mean(),
        perbandingan_model_VALIDASI["RMSE_Iterasi2_Val"].mean(),
        perbandingan_model_VALIDASI["RMSE_Iterasi3_Val"].mean(),
    ],
    "Average_MAE": [
        perbandingan_model_VALIDASI["MAE_Baseline_Val"].mean(),
        perbandingan_model_VALIDASI["MAE_FineTuning_Val"].mean(),
        perbandingan_model_VALIDASI["MAE_Iterasi1_Val"].mean(),
        perbandingan_model_VALIDASI["MAE_Iterasi2_Val"].mean(),
        perbandingan_model_VALIDASI["MAE_Iterasi3_Val"].mean(),
    ],
    "Average_MAPE": [
        perbandingan_model_VALIDASI["MAPE_Baseline_Val"].mean(),
        perbandingan_model_VALIDASI["MAPE_FineTuning_Val"].mean(),
        perbandingan_model_VALIDASI["MAPE_Iterasi1_Val"].mean(),
        perbandingan_model_VALIDASI["MAPE_Iterasi2_Val"].mean(),
        perbandingan_model_VALIDASI["MAPE_Iterasi3_Val"].mean(),
    ],
})

ringkasan_model_validasi = ringkasan_model_validasi.sort_values(by="Average_RMSE", ascending=True).reset_index(drop=True)

print("Ringkasan Rata-Rata Performa Seluruh Model (VALIDASI -- dasar pemilihan model_terbaik):")
ringkasan_model_validasi

In [ ]:
# ============================================================
# KEGIATAN 4: Menentukan Model Terbaik Berdasarkan Rata-Rata RMSE Validasi Terkecil
# [TASK GROUP 2] Sebelumnya berdasarkan ringkasan_model (test 2025) --
# sekarang berdasarkan ringkasan_model_validasi supaya data test 2025 tidak
# ikut menentukan pemenang (lihat catatan di atas). MAE dan MAPE (validasi)
# digunakan sebagai informasi pendukung; angka test 2025 dicetak terpisah
# di bawah sebagai KONFIRMASI, bukan alasan pemilihan.
# ============================================================

# Model terbaik adalah baris pertama setelah diurutkan berdasarkan Average_RMSE VALIDASI (ascending)
model_terbaik = ringkasan_model_validasi.iloc[0]["Model"]

rmse_terbaik = ringkasan_model_validasi.iloc[0]["Average_RMSE"]
mae_pendukung = ringkasan_model_validasi.iloc[0]["Average_MAE"]
mape_pendukung = ringkasan_model_validasi.iloc[0]["Average_MAPE"]

# Angka test 2025 untuk model yang sama, dicetak sebagai konfirmasi akhir (bukan dasar pemilihan)
baris_test_konfirmasi = ringkasan_model[ringkasan_model["Model"] == model_terbaik].iloc[0]

print(f"Model Terbaik (berdasarkan Average RMSE VALIDASI terkecil): {model_terbaik}")
print(f"  - Average RMSE (Validasi) : {rmse_terbaik:.3f}")
print(f"  - Average MAE  (Validasi) : {mae_pendukung:.3f}  (informasi pendukung)")
print(f"  - Average MAPE (Validasi) : {mape_pendukung:.2f}%  (informasi pendukung)")
print(f"  - Average RMSE (Test 2025, KONFIRMASI akhir) : {baris_test_konfirmasi['Average_RMSE']:.3f}")

In [ ]:
# ============================================================
# KEGIATAN 5: Menentukan Model Terbaik untuk Setiap Jenis PLT
# (Untuk melihat apakah model terbaik berbeda antar jenis pembangkit)
# [TASK GROUP 2] Pemilihan sekarang berdasarkan RMSE VALIDASI per PLT
# (bukan RMSE test 2025) -- RMSE test 2025 tetap dicantumkan sebagai
# kolom KONFIRMASI terpisah untuk pelaporan akhir di Bab IV.
# ============================================================

kolom_rmse_pm = ["RMSE_Baseline", "RMSE_FineTuning", "RMSE_Iterasi1", "RMSE_Iterasi2", "RMSE_Iterasi3"]
peta_nama_model_pm = {
    "RMSE_Baseline": "Baseline", "RMSE_FineTuning": "FineTuning",
    "RMSE_Iterasi1": "Iterasi1", "RMSE_Iterasi2": "Iterasi2", "RMSE_Iterasi3": "Iterasi3"
}

kolom_rmse_val_pm = [
    "RMSE_Baseline_Val", "RMSE_FineTuning_Val", "RMSE_Iterasi1_Val",
    "RMSE_Iterasi2_Val", "RMSE_Iterasi3_Val",
]
peta_nama_model_val_pm = {
    "RMSE_Baseline_Val": "Baseline", "RMSE_FineTuning_Val": "FineTuning",
    "RMSE_Iterasi1_Val": "Iterasi1", "RMSE_Iterasi2_Val": "Iterasi2", "RMSE_Iterasi3_Val": "Iterasi3"
}
# Nama kolom RMSE test 2025 yang jadi KONFIRMASI akhir, per nama model_terbaik
peta_kolom_rmse_test = {
    "Baseline": "RMSE_Baseline", "FineTuning": "RMSE_FineTuning",
    "Iterasi1": "RMSE_Iterasi1", "Iterasi2": "RMSE_Iterasi2", "Iterasi3": "RMSE_Iterasi3",
}

model_terbaik_per_plt = perbandingan_model_VALIDASI[["Jenis_PLT"] + kolom_rmse_val_pm].copy()
model_terbaik_per_plt["Model_Terbaik"] = model_terbaik_per_plt[kolom_rmse_val_pm].idxmin(axis=1).map(
    peta_nama_model_val_pm
)
model_terbaik_per_plt["RMSE_Validasi_Terbaik"] = model_terbaik_per_plt[kolom_rmse_val_pm].min(axis=1)

# Mengambil RMSE test 2025 milik tahap pemenang (perbandingan_model, test-based)
# sebagai angka KONFIRMASI akhir -- BUKAN dasar pemilihan.
perbandingan_model_indexed = perbandingan_model.set_index("Jenis_PLT")

def _ambil_rmse_test_konfirmasi(baris):
    kolom_test = peta_kolom_rmse_test[baris["Model_Terbaik"]]
    return perbandingan_model_indexed.loc[baris["Jenis_PLT"], kolom_test]

model_terbaik_per_plt["RMSE_Test_2025_Konfirmasi"] = model_terbaik_per_plt.apply(
    _ambil_rmse_test_konfirmasi, axis=1
)

model_terbaik_per_plt = model_terbaik_per_plt[
    ["Jenis_PLT", "Model_Terbaik", "RMSE_Validasi_Terbaik", "RMSE_Test_2025_Konfirmasi"]
].sort_values(by="RMSE_Validasi_Terbaik", ascending=True).reset_index(drop=True)

print("Model Terbaik untuk Setiap Jenis PLT (berdasarkan RMSE VALIDASI terkecil; RMSE test 2025 = konfirmasi akhir):")
model_terbaik_per_plt

In [ ]:
# ============================================================
# KEGIATAN 6: Diagram Batang Perbandingan Rata-Rata RMSE, MAE, MAPE
# untuk Seluruh Model
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

warna_bar = sns.color_palette("viridis", len(ringkasan_model))

axes[0].bar(ringkasan_model["Model"], ringkasan_model["Average_RMSE"], color=warna_bar)
axes[0].set_title("Perbandingan Average RMSE")
axes[0].set_ylabel("RMSE")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(ringkasan_model["Model"], ringkasan_model["Average_MAE"], color=warna_bar)
axes[1].set_title("Perbandingan Average MAE")
axes[1].set_ylabel("MAE")
axes[1].tick_params(axis="x", rotation=30)

axes[2].bar(ringkasan_model["Model"], ringkasan_model["Average_MAPE"], color=warna_bar)
axes[2].set_title("Perbandingan Average MAPE")
axes[2].set_ylabel("MAPE (%)")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 7: Diagram Batang Perbandingan RMSE Setiap Jenis PLT
# pada Seluruh Model (Baseline, Fine-Tuning, Iterasi 1-3)
# ============================================================

# Menyusun ulang data ke bentuk long-format agar mudah dibuat grouped bar chart
df_rmse_long = perbandingan_model.melt(
    id_vars="Jenis_PLT",
    value_vars=kolom_rmse_pm,
    var_name="Model",
    value_name="RMSE"
)
df_rmse_long["Model"] = df_rmse_long["Model"].map(peta_nama_model_pm)

plt.figure(figsize=(14, 6))
sns.barplot(data=df_rmse_long, x="Jenis_PLT", y="RMSE", hue="Model", palette="viridis")
plt.title("Perbandingan RMSE Setiap Jenis PLT pada Seluruh Model")
plt.xlabel("Jenis PLT")
plt.ylabel("RMSE")
plt.xticks(rotation=30)
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 8: Menyusun Konfigurasi Hyperparameter Model Terbaik
# ============================================================

# Dictionary konfigurasi setiap model, digunakan untuk mengambil konfigurasi model_terbaik
daftar_konfigurasi_model = {
    "Baseline": {
        "Metode": "Direct Training (per jenis PLT)",
        "Window_Size": WINDOW_SIZE,
        "Learning_Rate": "default (Adam)",
        "Dropout": 0.2,
        "Batch_Size": BATCH_SIZE,
        "Epochs": EPOCHS,
        "Optimizer": "Adam",
        "Callback": "EarlyStopping, ReduceLROnPlateau",
    },
    "FineTuning": {
        "Metode": "Transfer Learning (PLTA,PLTB,PLTM,PLTMH,PLTS) / Direct Training (PLTS Atap, PLT Hybrid)",
        "Window_Size": WINDOW_SIZE,
        "Learning_Rate": LR_FINETUNE,
        "Dropout": 0.2,
        "Batch_Size": BATCH_SIZE_FINETUNE,
        "Epochs": EPOCHS_FINETUNE,
        "Optimizer": "Adam",
        "Callback": "EarlyStopping, ReduceLROnPlateau",
    },
    "Iterasi1": {
        "Metode": "Transfer Learning (PLTA,PLTB,PLTM,PLTMH,PLTS) / Direct Training (PLTS Atap, PLT Hybrid)",
        "Window_Size": WINDOW_SIZE,
        "Learning_Rate": LR_ITER1,
        "Dropout": 0.2,
        "Batch_Size": BATCH_SIZE_ITER1,
        "Epochs": EPOCHS_ITER1,
        "Optimizer": "Adam",
        "Callback": "EarlyStopping, ReduceLROnPlateau",
    },
    "Iterasi2": {
        "Metode": "Transfer Learning (PLTA,PLTB,PLTM,PLTMH,PLTS) / Direct Training (PLTS Atap, PLT Hybrid)",
        "Window_Size": WINDOW_SIZE_ITER2,
        "Learning_Rate": LR_ITER2,
        "Dropout": 0.2,
        "Batch_Size": BATCH_SIZE_ITER2,
        "Epochs": EPOCHS_ITER2,
        "Optimizer": "Adam",
        "Callback": "EarlyStopping, ReduceLROnPlateau",
    },
    "Iterasi3": {
        "Metode": "Transfer Learning (PLTA,PLTB,PLTM,PLTMH,PLTS) / Direct Training (PLTS Atap, PLT Hybrid)",
        "Window_Size": WINDOW_SIZE_ITER3,
        "Learning_Rate": LR_ITER3,
        "Dropout": DROPOUT_ITER3,
        "Batch_Size": BATCH_SIZE_ITER3,
        "Epochs": EPOCHS_ITER3,
        "Optimizer": "Adam",
        "Callback": "EarlyStopping, ReduceLROnPlateau",
    },
}

# Mengambil konfigurasi sesuai model_terbaik yang telah ditentukan pada Kegiatan 4
konfigurasi_model_terbaik = daftar_konfigurasi_model[model_terbaik]

print(f"Konfigurasi Hyperparameter Model Terbaik ({model_terbaik}):")
for key, value in konfigurasi_model_terbaik.items():
    print(f"  - {key:15s}: {value}")

In [ ]:
# ============================================================
# KEGIATAN 9: Kesimpulan Otomatis - Model Terbaik Penelitian
# ============================================================

print("=" * 60)
print("KESIMPULAN: MODEL TERBAIK PENELITIAN")
print("=" * 60)

print(f"\nModel yang dipilih sebagai Model Terbaik Penelitian adalah: {model_terbaik}")
print(f"\nAlasan pemilihan:")
print(f"  1. {model_terbaik} memiliki rata-rata RMSE terkecil di antara seluruh model")
print(f"     yang diuji, yaitu {rmse_terbaik:.3f} (kriteria utama pemilihan).")
print(f"  2. Sebagai informasi pendukung, {model_terbaik} memiliki rata-rata MAE sebesar")
print(f"     {mae_pendukung:.3f} dan rata-rata MAPE sebesar {mape_pendukung:.2f}%.")

# Menampilkan perbandingan singkat terhadap Baseline (jika model terbaik bukan Baseline)
if model_terbaik != "Baseline":
    rmse_baseline_avg = ringkasan_model[ringkasan_model["Model"] == "Baseline"]["Average_RMSE"].values[0]
    penurunan_persen = (rmse_baseline_avg - rmse_terbaik) / rmse_baseline_avg * 100
    print(f"  3. Dibandingkan dengan Baseline LSTM (Average RMSE = {rmse_baseline_avg:.3f}),")
    print(f"     model ini menurunkan RMSE sebesar {penurunan_persen:.2f}%.")

print(f"\nKonfigurasi hyperparameter model terbaik ({model_terbaik}):")
for key, value in konfigurasi_model_terbaik.items():
    print(f"  - {key:15s}: {value}")

print("\nCatatan: hasil ini akan digunakan sebagai acuan konfigurasi pada tahap")
print("pelatihan Model Final (belum dilakukan pada tahap ini).")
print("=" * 60)

In [ ]:
# ============================================================
# KEGIATAN 0: Import Library Tambahan untuk Penyimpanan Model
# (os, pickle, json, datetime belum digunakan pada tahap sebelumnya)
# ============================================================

import os
import pickle
import json
from datetime import datetime

# Membuat folder model_final/ jika belum ada
FOLDER_MODEL_FINAL = "model_final"
os.makedirs(FOLDER_MODEL_FINAL, exist_ok=True)

print(f"Folder '{FOLDER_MODEL_FINAL}/' siap digunakan untuk menyimpan model, scaler, dan metadata.")

In [ ]:
# ============================================================
# KEGIATAN 1: Menampilkan Tabel Konfigurasi Model Terbaik per Jenis PLT
# ============================================================

print("Model terbaik untuk setiap jenis PLT (hasil tahap Penentuan Model Terbaik):")
display(model_terbaik_per_plt)

# [TASK GROUP 3] Metode pelatihan final SEKARANG ditentukan PER PLT dari
# model_terbaik_per_plt (hasil Task Group 2, berbasis RMSE validasi) --
# sebelumnya METODE_FINAL_PER_PLT adalah dict hard-coded (snapshot manual
# dari run lama) yang tidak lagi cocok dengan model_terbaik_per_plt saat
# ini. Aturan turunannya:
#   - Kalau Model_Terbaik PLT itu "Baseline" -> selalu "Direct Training"
#     (tahap Baseline memang tidak pernah memuat bobot Pre-Training).
#   - Selain itu -> "Transfer Learning" HANYA jika PLT-nya ada di
#     PLT_TRANSFER_LEARNING (punya data Pre-Training nasional); PLTS Atap
#     dan PLT Hybrid selalu "Direct Training" apa pun tahap pemenangnya.
METODE_FINAL_PER_PLT = {}
for _, baris_terbaik in model_terbaik_per_plt.iterrows():
    plt_name = baris_terbaik["Jenis_PLT"]
    stage_menang = baris_terbaik["Model_Terbaik"]
    if stage_menang == "Baseline":
        METODE_FINAL_PER_PLT[plt_name] = "Direct Training"
    elif plt_name in PLT_TRANSFER_LEARNING:
        METODE_FINAL_PER_PLT[plt_name] = "Transfer Learning"
    else:
        METODE_FINAL_PER_PLT[plt_name] = "Direct Training"

print("\nMetode pelatihan Model Final per jenis PLT (diturunkan dari model_terbaik_per_plt):")
for plt_name, metode in METODE_FINAL_PER_PLT.items():
    print(f"  - {plt_name:12s}: {metode}")

# [TASK GROUP 3] Konfigurasi hyperparameter final PER PLT -- setiap PLT
# memakai konfigurasi dari TAHAP yang menang untuk PLT itu sendiri
# (model_terbaik_per_plt), bukan lagi satu konfigurasi_model_terbaik global
# yang dipaksakan ke seluruh PLT.
konfigurasi_terbaik_per_plt = {}
for _, baris_terbaik in model_terbaik_per_plt.iterrows():
    plt_name = baris_terbaik["Jenis_PLT"]
    stage_menang = baris_terbaik["Model_Terbaik"]
    konfigurasi_terbaik_per_plt[plt_name] = daftar_konfigurasi_model[stage_menang]

print("\nKonfigurasi hyperparameter Model Final per jenis PLT:")
for plt_name, konfigurasi in konfigurasi_terbaik_per_plt.items():
    print(f"  - {plt_name:12s}: {konfigurasi}")

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# ============================================================
# KEGIATAN 2: Preprocessing Data Regional Lengkap (2023-2025)
# Seluruh data digunakan sebagai data pelatihan, TANPA menyisakan data uji
# ============================================================

# df_regional sudah tersedia (hasil Kegiatan 1 & 2 tahap Fine-Tuning), diurutkan berdasarkan
# jenis PLT dan tanggal. Kolom fitur dan target juga sama seperti tahap sebelumnya.
print("Menggunakan df_regional yang sudah dipreprocessing (urut jenis PLT & tanggal).")
print(f"Fitur input : {FEATURE_COLS_REGIONAL}")
print(f"Target      : {TARGET_COL_REGIONAL}")

# Normalisasi ulang menggunakan MinMaxScaler KHUSUS untuk Model Final
# (scaler baru dibuat dari SELURUH data 2023-2025, bukan scaler dari tahap Fine-Tuning
#  yang sebelumnya hanya fit pada periode training 2023-2024)
scaler_fitur_final_per_plt = {}
scaler_target_final_per_plt = {}
data_scaled_final_per_plt = {}

for plt_name in METODE_FINAL_PER_PLT.keys():
    # Fix: Change 'jenis_plt' to 'Jenis' (capital 'J')
    data_plt = df_regional[df_regional["Jenis"] == plt_name].copy()

    fitur_plt = data_plt[FEATURE_COLS_REGIONAL].values
    target_plt = data_plt[[TARGET_COL_REGIONAL]].values

    scaler_fitur = MinMaxScaler(feature_range=(0, 1))
    scaler_target = MinMaxScaler(feature_range=(0, 1))

    fitur_scaled = scaler_fitur.fit_transform(fitur_plt)
    target_scaled = scaler_target.fit_transform(target_plt)

    data_gabungan_scaled = np.hstack([fitur_scaled, target_scaled])

    scaler_fitur_final_per_plt[plt_name] = scaler_fitur
    scaler_target_final_per_plt[plt_name] = scaler_target
    data_scaled_final_per_plt[plt_name] = data_gabungan_scaled

    print(f"{plt_name:12s} -> Total data (2023-2025): {len(data_plt)}")

In [ ]:
# ============================================================
# KEGIATAN 3: Membentuk Data Sequence dengan create_sequences()
# (Window size disesuaikan per jenis PLT berdasarkan konfigurasi terbaik PLT itu sendiri)
# ============================================================

# [TASK GROUP 3] Window size sekarang diambil PER PLT dari
# konfigurasi_terbaik_per_plt (konfigurasi tahap yang menang untuk PLT itu
# sendiri) -- sebelumnya satu WINDOW_SIZE_FINAL global dipakai untuk semua
# PLT, padahal mis. Iterasi2 pakai window 12 bulan, bukan 6.
sequence_final_per_plt = {}

for plt_name in METODE_FINAL_PER_PLT.keys():
    data_gabungan_scaled = data_scaled_final_per_plt[plt_name]
    window_size_plt = konfigurasi_terbaik_per_plt[plt_name]["Window_Size"]

    X_seq_final, y_seq_final = create_sequences(data_gabungan_scaled, window_size=window_size_plt)

    # Mengambil kolom target (Produksi) saja, yaitu kolom paling akhir
    y_target_final = y_seq_final[:, -1]

    sequence_final_per_plt[plt_name] = {
        "X": X_seq_final,
        "y": y_target_final
    }

    print(f"{plt_name:12s} -> Total sequence (window={window_size_plt}) untuk pelatihan final: {len(X_seq_final)}")

In [ ]:
# ============================================================
# KEGIATAN 4: Melatih Model Final - Transfer Learning
# Berlaku untuk: PLTA, PLTB, PLTMH, PLTS
# (Memuat bobot Pre-Training, dilanjutkan dengan seluruh data regional 2023-2025)
# ============================================================

model_final_per_plt = {}

PLT_FINAL_TRANSFER_LEARNING = [p for p, m in METODE_FINAL_PER_PLT.items() if m == "Transfer Learning"]

N_FEATURES_FINAL = len(FEATURE_COLS_REGIONAL) + 1  # Cuaca, Kapasitas, Produksi historis

for plt_name in PLT_FINAL_TRANSFER_LEARNING:
    print(f"\n=== Melatih Model Final (Transfer Learning) untuk: {plt_name} ===")

    # [TASK GROUP 3] Konfigurasi (window_size, dropout, learning_rate, epochs,
    # batch_size) sekarang diambil PER PLT dari konfigurasi_terbaik_per_plt,
    # bukan lagi konfigurasi_model_terbaik global yang sama untuk semua PLT.
    konfigurasi_plt = konfigurasi_terbaik_per_plt[plt_name]

    X_final = sequence_final_per_plt[plt_name]["X"]
    y_final = sequence_final_per_plt[plt_name]["y"]

    # 1. Membangun arsitektur sesuai konfigurasi terbaik PLT ini (sama seperti Baseline, dropout sesuai konfigurasi)
    model_final = build_baseline_lstm(
        window_size=konfigurasi_plt["Window_Size"],
        n_features=N_FEATURES_FINAL,
        dropout_rate=konfigurasi_plt["Dropout"]
    )

    # 2. Memuat bobot hasil Pre-Training Nasional sebagai bobot awal
    bobot_pretrain = hasil_pretrain_per_plt[plt_name]["model"].get_weights()
    model_final.set_weights(bobot_pretrain)

    # 3. Compile dengan learning rate sesuai konfigurasi terbaik PLT ini
    current_learning_rate = konfigurasi_plt["Learning_Rate"]
    if isinstance(current_learning_rate, str) and current_learning_rate == "default (Adam)":
        # Use Keras default for Adam, which is 0.001
        optimizer_lr = 0.001
    else:
        optimizer_lr = current_learning_rate

    model_final.compile(
        optimizer=Adam(learning_rate=optimizer_lr),
        loss="mse", metrics=["mae"]
    )

    # 4. Melatih model menggunakan SELURUH data regional (tanpa validation_data / split test)
    model_final.fit(
        X_final, y_final,
        epochs=konfigurasi_plt["Epochs"],
        batch_size=konfigurasi_plt["Batch_Size"],
        callbacks=get_callbacks(),
        verbose=0
    )

    model_final_per_plt[plt_name] = {
        "model": model_final,
        "metode": "Transfer Learning"
    }

    print(f"Pelatihan Model Final untuk {plt_name} selesai.")

In [ ]:
# ============================================================
# KEGIATAN 5: Melatih Model Final - Direct Training
# Berlaku untuk: PLTM, PLTS Atap, PLT Hybrid
# (Baseline LSTM dari nol menggunakan seluruh data regional 2023-2025)
# ============================================================

PLT_FINAL_DIRECT_TRAINING = [p for p, m in METODE_FINAL_PER_PLT.items() if m == "Direct Training"]

for plt_name in PLT_FINAL_DIRECT_TRAINING:
    print(f"\n=== Melatih Model Final (Direct Training) untuk: {plt_name} ===")

    # [TASK GROUP 3] Konfigurasi PER PLT, sama seperti loop Transfer Learning di atas.
    konfigurasi_plt = konfigurasi_terbaik_per_plt[plt_name]

    X_final = sequence_final_per_plt[plt_name]["X"]
    y_final = sequence_final_per_plt[plt_name]["y"]

    # Arsitektur sama seperti Baseline, dilatih dari nol (tanpa bobot Pre-Training)
    model_final_direct = build_baseline_lstm(
        window_size=konfigurasi_plt["Window_Size"],
        n_features=N_FEATURES_FINAL,
        dropout_rate=konfigurasi_plt["Dropout"]
    )
    current_learning_rate = konfigurasi_plt["Learning_Rate"]
    if isinstance(current_learning_rate, str) and current_learning_rate == "default (Adam)":
        optimizer_lr = 0.001  # Keras default for Adam
    else:
        optimizer_lr = current_learning_rate

    model_final_direct.compile(
        optimizer=Adam(learning_rate=optimizer_lr),
        loss="mse", metrics=["mae"]
    )

    model_final_direct.fit(
        X_final, y_final,
        epochs=konfigurasi_plt["Epochs"],
        batch_size=konfigurasi_plt["Batch_Size"],
        callbacks=get_callbacks(),
        verbose=0
    )

    model_final_per_plt[plt_name] = {
        "model": model_final_direct,
        "metode": "Direct Training"
    }

    print(f"Pelatihan Model Final untuk {plt_name} selesai.")

In [ ]:
# ============================================================
# KEGIATAN 6: Menyimpan Model Final ke Format .keras
# ============================================================

for plt_name, info in model_final_per_plt.items():
    # Nama file mengganti spasi dengan underscore agar konsisten (PLTS Atap -> PLTS_Atap)
    nama_file_aman = plt_name.replace(" ", "_")
    path_model = os.path.join(FOLDER_MODEL_FINAL, f"model_final_{nama_file_aman}.keras")

    info["model"].save(path_model)
    info["path_model"] = path_model

    print(f"Model {plt_name:12s} disimpan ke: {path_model}")

In [ ]:
# ============================================================
# KEGIATAN 7: Menyimpan Scaler Input dan Target ke Format .pkl
# ============================================================

for plt_name in METODE_FINAL_PER_PLT.keys():
    nama_file_aman = plt_name.replace(" ", "_")

    path_scaler_fitur = os.path.join(FOLDER_MODEL_FINAL, f"scaler_fitur_{nama_file_aman}.pkl")
    path_scaler_target = os.path.join(FOLDER_MODEL_FINAL, f"scaler_target_{nama_file_aman}.pkl")

    with open(path_scaler_fitur, "wb") as f:
        pickle.dump(scaler_fitur_final_per_plt[plt_name], f)

    with open(path_scaler_target, "wb") as f:
        pickle.dump(scaler_target_final_per_plt[plt_name], f)

    print(f"Scaler {plt_name:12s} disimpan ke: {path_scaler_fitur} & {path_scaler_target}")

In [ ]:
# ============================================================
# KEGIATAN 8: Menyimpan Metadata Model ke metadata_model.json
# ============================================================

metadata_model = {}

for plt_name, info in model_final_per_plt.items():
    # [TASK GROUP 3] Metadata sekarang mencerminkan konfigurasi PER PLT.
    konfigurasi_plt = konfigurasi_terbaik_per_plt[plt_name]
    metadata_model[plt_name] = {
        "jenis_plt": plt_name,
        "metode_pelatihan": info["metode"],
        "window_size": konfigurasi_plt["Window_Size"],
        "learning_rate": konfigurasi_plt["Learning_Rate"],
        "dropout": konfigurasi_plt["Dropout"],
        "batch_size": konfigurasi_plt["Batch_Size"],
        "epochs": konfigurasi_plt["Epochs"],
        "optimizer": konfigurasi_plt["Optimizer"],
        "fitur_input": FEATURE_COLS_REGIONAL + [TARGET_COL_REGIONAL],  # termasuk produksi historis sbg fitur
        "target_prediksi": TARGET_COL_REGIONAL,
        "tanggal_pelatihan": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

path_metadata = os.path.join(FOLDER_MODEL_FINAL, "metadata_model.json")
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata_model, f, indent=4, ensure_ascii=False)

print(f"Metadata seluruh model disimpan ke: {path_metadata}")

In [ ]:
# ============================================================
# KEGIATAN 9: Menampilkan Ringkasan Seluruh Model Final
# Beserta Lokasi Penyimpanannya
# ============================================================

ringkasan_model_final = pd.DataFrame([
    {
        "Jenis_PLT": plt_name,
        "Metode": info["metode"],
        "Path_Model": info["path_model"],
        "Path_Scaler_Fitur": os.path.join(FOLDER_MODEL_FINAL, f"scaler_fitur_{plt_name.replace(' ', '_')}.pkl"),
        "Path_Scaler_Target": os.path.join(FOLDER_MODEL_FINAL, f"scaler_target_{plt_name.replace(' ', '_')}.pkl"),
    }
    for plt_name, info in model_final_per_plt.items()
])

print("Ringkasan Seluruh Model Final:")
display(ringkasan_model_final)

print(f"\nSeluruh model, scaler, dan metadata telah disimpan pada folder: '{FOLDER_MODEL_FINAL}/'")
print(f"Total model final yang berhasil dibuat: {len(model_final_per_plt)}")

In [ ]:
# ============================================================
# KEGIATAN 0: Import Library Tambahan dan Menyiapkan Folder Output
# (load_model, glob belum digunakan pada tahap sebelumnya)
# ============================================================

from tensorflow.keras.models import load_model
import glob

FOLDER_MODEL_FINAL = "model_final"
FOLDER_FORECAST = "forecast"
os.makedirs(FOLDER_FORECAST, exist_ok=True)

# Menentukan periode forecasting: Januari 2026 - Desember 2028 (36 bulan)
PERIODE_FORECAST = pd.date_range(start="2026-01-01", end="2028-12-01", freq="MS")

print(f"Periode forecasting: {PERIODE_FORECAST[0].strftime('%B %Y')} - {PERIODE_FORECAST[-1].strftime('%B %Y')}")
print(f"Total bulan yang diprediksi: {len(PERIODE_FORECAST)} bulan")

import os
import json

In [ ]:
# ============================================================
# KEGIATAN 1: Memuat Metadata Model dari metadata_model.json
# ============================================================

FOLDER_MODEL_FINAL = "model_final"

path_metadata = os.path.join(FOLDER_MODEL_FINAL, "metadata_model.json")

with open(path_metadata, "r", encoding="utf-8") as f:
    metadata_model = json.load(f)

print("Metadata model berhasil dimuat untuk jenis PLT:")
for plt_name, meta in metadata_model.items():
    print(f"  - {plt_name:12s} | Metode: {meta['metode_pelatihan']:18s} | Window: {meta['window_size']} bulan")

In [ ]:
# ============================================================
# KEGIATAN 2: Memuat Model Final (.keras) dan Scaler (.pkl)
# untuk Setiap Jenis PLT
# ============================================================

model_final_loaded = {}
scaler_fitur_loaded = {}
scaler_target_loaded = {}

for plt_name in metadata_model.keys():
    nama_file_aman = plt_name.replace(" ", "_")

    path_model = os.path.join(FOLDER_MODEL_FINAL, f"model_final_{nama_file_aman}.keras")
    path_scaler_fitur = os.path.join(FOLDER_MODEL_FINAL, f"scaler_fitur_{nama_file_aman}.pkl")
    path_scaler_target = os.path.join(FOLDER_MODEL_FINAL, f"scaler_target_{nama_file_aman}.pkl")

    model_final_loaded[plt_name] = load_model(path_model)

    with open(path_scaler_fitur, "rb") as f:
        scaler_fitur_loaded[plt_name] = pickle.load(f)

    with open(path_scaler_target, "rb") as f:
        scaler_target_loaded[plt_name] = pickle.load(f)

    print(f"Model, scaler fitur, dan scaler target untuk {plt_name} berhasil dimuat.")

In [ ]:
# ============================================================
# KEGIATAN 3: Menyiapkan Sequence Awal (Seed) untuk Forecasting
# Menggunakan Data Historis 2023-2025 sesuai Window Size Masing-Masing Model
# ============================================================

# df_regional (sudah diurutkan berdasarkan jenis_plt & tanggal) digunakan kembali sebagai basis historis
seed_data_per_plt = {}

for plt_name, meta in metadata_model.items():
    window_size_plt = meta["window_size"]

    data_plt = df_regional[df_regional["Jenis"] == plt_name].copy()
    data_plt = data_plt.sort_values("Tanggal").reset_index(drop=True)

    # Mengambil window_size bulan TERAKHIR dari data historis sebagai titik awal forecasting
    fitur_terakhir = data_plt[FEATURE_COLS_REGIONAL].values[-window_size_plt:]
    target_terakhir = data_plt[[TARGET_COL_REGIONAL]].values[-window_size_plt:]

    # Menormalisasi menggunakan scaler yang sama seperti saat pelatihan Model Final
    fitur_scaled = scaler_fitur_loaded[plt_name].transform(fitur_terakhir)
    target_scaled = scaler_target_loaded[plt_name].transform(target_terakhir)

    seed_sequence = np.hstack([fitur_scaled, target_scaled])  # shape: (window_size, n_fitur+1)

    seed_data_per_plt[plt_name] = {
        "seed_sequence": seed_sequence,
        # Rata-rata fitur (Cuaca, Kapasitas) SKALA ASLI pada 12 bulan terakhir,
        # digunakan sebagai asumsi LOCF karena data cuaca/kapasitas 2026-2028 belum tersedia.
        # Asumsi ini mencerminkan pola musiman rata-rata terakhir, bukan nilai statis 1 bulan.
        "fitur_locf_asli": data_plt[FEATURE_COLS_REGIONAL].values[-12:].mean(axis=0)
    }

    print(f"{plt_name:12s} -> Seed sequence disiapkan (window={window_size_plt} bulan).")

In [ ]:
# ============================================================
# KEGIATAN 4: Membuat Fungsi Forecasting Autoregressive (Multi-Step)
# ============================================================

def forecast_autoregressive(model, seed_sequence, fitur_locf_asli, scaler_fitur, scaler_target,
                             n_steps, window_size):
    """
    Melakukan forecasting multi-step secara autoregressive.

    Pada setiap langkah:
      1. Model memprediksi 1 nilai produksi berikutnya berdasarkan window_size data terakhir.
      2. Fitur (Cuaca, Kapasitas) untuk bulan prediksi menggunakan asumsi LOCF
         (rata-rata musiman 12 bulan terakhir data historis).
      3. Hasil prediksi (target) digabungkan dengan fitur LOCF, lalu dimasukkan kembali
         ke dalam sequence sebagai input untuk memprediksi bulan berikutnya (sliding window).

    Returns
    -------
    list berisi n_steps nilai prediksi (masih dalam skala normalisasi)
    """
    current_sequence = seed_sequence.copy()
    hasil_prediksi_scaled = []

    # Menormalisasi fitur LOCF sekali di awal (fitur diasumsikan konstan sepanjang periode forecast)
    fitur_locf_scaled = scaler_fitur.transform(fitur_locf_asli.reshape(1, -1))[0]

    for step in range(n_steps):
        # Reshape sequence menjadi bentuk input model: (1, window_size, n_fitur)
        input_model = current_sequence.reshape(1, window_size, -1)

        # Prediksi 1 langkah ke depan (masih skala normalisasi)
        pred_scaled = model.predict(input_model, verbose=0)[0, 0]
        hasil_prediksi_scaled.append(pred_scaled)

        # Membentuk baris data baru: fitur LOCF (scaled) + hasil prediksi (scaled)
        baris_baru = np.append(fitur_locf_scaled, pred_scaled)

        # Sliding window: buang baris paling awal, tambahkan baris baru di akhir
        current_sequence = np.vstack([current_sequence[1:], baris_baru])

    return hasil_prediksi_scaled

print("Fungsi forecast_autoregressive() berhasil dibuat.")

In [ ]:
# ============================================================
# KEGIATAN 5: Menjalankan Forecasting Autoregressive untuk Seluruh Jenis PLT
# Periode: Januari 2026 - Desember 2028 (36 bulan)
# ============================================================

hasil_forecast_per_plt = {}

for plt_name, meta in metadata_model.items():
    print(f"Menjalankan forecasting untuk: {plt_name} ...")

    window_size_plt = meta["window_size"]
    model = model_final_loaded[plt_name]
    seed_sequence = seed_data_per_plt[plt_name]["seed_sequence"]
    fitur_locf_asli = seed_data_per_plt[plt_name]["fitur_locf_asli"]

    # Forecasting 36 langkah ke depan (skala normalisasi)
    pred_scaled_list = forecast_autoregressive(
        model=model,
        seed_sequence=seed_sequence,
        fitur_locf_asli=fitur_locf_asli,
        scaler_fitur=scaler_fitur_loaded[plt_name],
        scaler_target=scaler_target_loaded[plt_name],
        n_steps=len(PERIODE_FORECAST),
        window_size=window_size_plt
    )

    hasil_forecast_per_plt[plt_name] = pred_scaled_list

print("\nForecasting untuk seluruh jenis PLT selesai dijalankan.")

In [ ]:
# ============================================================
# KEGIATAN 6: Inverse Transform Hasil Forecasting ke Satuan Produksi Asli
# ============================================================

hasil_forecast_asli_per_plt = {}

for plt_name in metadata_model.keys():
    pred_scaled_array = np.array(hasil_forecast_per_plt[plt_name]).reshape(-1, 1)
    pred_asli = scaler_target_loaded[plt_name].inverse_transform(pred_scaled_array).flatten()

    # Produksi tidak boleh bernilai negatif secara fisis, sehingga dibatasi minimum 0
    pred_asli = np.clip(pred_asli, a_min=0, a_max=None)

    hasil_forecast_asli_per_plt[plt_name] = pred_asli

    print(f"{plt_name:12s} -> Inverse transform selesai. Contoh 3 bulan pertama: {np.round(pred_asli[:3], 2)}")

In [ ]:
# ============================================================
# KEGIATAN 7: Membentuk DataFrame Hasil Forecasting per Jenis PLT
# (Kolom: Tanggal, Tahun, Bulan, Jenis_PLT, Produksi_Prediksi)
# ============================================================

daftar_df_forecast_plt = []

for plt_name in metadata_model.keys():
    df_plt = pd.DataFrame({
        "Tanggal": PERIODE_FORECAST,
        "Tahun": PERIODE_FORECAST.year,
        "Bulan": PERIODE_FORECAST.month,
        "Jenis_PLT": plt_name,
        "Produksi_Prediksi": hasil_forecast_asli_per_plt[plt_name]
    })
    daftar_df_forecast_plt.append(df_plt)

print("DataFrame forecasting per jenis PLT berhasil dibentuk untuk seluruh PLT.")

In [ ]:
# ============================================================
# KEGIATAN 8: Menggabungkan Seluruh Hasil Forecasting ke DataFrame forecast_ebt
# ============================================================

forecast_ebt = pd.concat(daftar_df_forecast_plt, ignore_index=True)
forecast_ebt = forecast_ebt.sort_values(by=["Jenis_PLT", "Tanggal"]).reset_index(drop=True)

print("DataFrame forecast_ebt (seluruh jenis PLT, 2026-2028):")
forecast_ebt

In [ ]:
# ============================================================
# KEGIATAN 9: Membentuk DataFrame Total Produksi EBT Setiap Bulan
# (Menjumlahkan seluruh jenis PLT)
# ============================================================

forecast_total = forecast_ebt.groupby(["Tanggal", "Tahun", "Bulan"], as_index=False)["Produksi_Prediksi"] \
                              .sum().rename(columns={"Produksi_Prediksi": "Total_Produksi_EBT"})

forecast_total = forecast_total.sort_values(by="Tanggal").reset_index(drop=True)

print("DataFrame forecast_total (total seluruh jenis PLT per bulan, 2026-2028):")
forecast_total

In [ ]:
# ============================================================
# KEGIATAN 10: Grafik Garis Hasil Prediksi Masing-Masing Jenis PLT
# ============================================================

plt.figure(figsize=(14, 6))

for plt_name in metadata_model.keys():
    data_plt_forecast = forecast_ebt[forecast_ebt["Jenis_PLT"] == plt_name]
    plt.plot(data_plt_forecast["Tanggal"], data_plt_forecast["Produksi_Prediksi"], marker="o",
             markersize=3, label=plt_name)

plt.title("Hasil Forecasting Produksi EBT per Jenis PLT (2026-2028)")
plt.xlabel("Periode")
plt.ylabel("Produksi (GWh)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 11: Grafik Total Produksi EBT Setiap Bulan
# ============================================================

plt.figure(figsize=(14, 5))
plt.plot(forecast_total["Tanggal"], forecast_total["Total_Produksi_EBT"],
         color="seagreen", marker="o", markersize=3)
plt.title("Total Forecasting Produksi EBT Provinsi Sulawesi Selatan (2026-2028)")
plt.xlabel("Periode")
plt.ylabel("Total Produksi (GWh)")
plt.tight_layout()
plt.show()

import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# KEGIATAN 12: Grafik Perbandingan Data Historis vs Hasil Forecasting
# per Jenis PLT (agar transisi pola data dapat diamati)
# ============================================================

n_plt_forecast = len(metadata_model)
n_cols = 2
n_rows = int(np.ceil(n_plt_forecast / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, plt_name in enumerate(metadata_model.keys()):
    # Data historis (2023-2025)
    data_historis = df_regional[df_regional["Jenis"] == plt_name].sort_values("Tanggal")

    # Data forecasting (2026-2028)
    data_forecast = forecast_ebt[forecast_ebt["Jenis_PLT"] == plt_name]

    axes[i].plot(data_historis["Tanggal"], data_historis[TARGET_COL_REGIONAL],
                 label="Historis (2023-2025)", color="steelblue")
    axes[i].plot(data_forecast["Tanggal"], data_forecast["Produksi_Prediksi"],
                 label="Forecasting (2026-2028)", color="darkorange", linestyle="--")
    axes[i].axvline(x=pd.Timestamp("2026-01-01"), color="gray", linestyle=":", alpha=0.7)
    axes[i].set_title(f"Historis vs Forecasting - {plt_name}")
    axes[i].set_xlabel("Periode")
    axes[i].set_ylabel("Produksi (GWh)")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# KEGIATAN 13: Menyimpan Hasil Forecasting
# (forecast_per_PLT.csv, forecast_total.csv, forecast_per_PLT.xlsx)
# ============================================================

path_csv_per_plt = os.path.join(FOLDER_FORECAST, "forecast_per_PLT.csv")
path_csv_total = os.path.join(FOLDER_FORECAST, "forecast_total.csv")
path_xlsx_per_plt = os.path.join(FOLDER_FORECAST, "forecast_per_PLT.xlsx")

forecast_ebt.to_csv(path_csv_per_plt, index=False)
forecast_total.to_csv(path_csv_total, index=False)
forecast_ebt.to_excel(path_xlsx_per_plt, index=False)

print("Hasil forecasting berhasil disimpan:")
print(f"  - {path_csv_per_plt}")
print(f"  - {path_csv_total}")
print(f"  - {path_xlsx_per_plt}")

In [ ]:
# ============================================================
# KEGIATAN 14: Ringkasan Forecasting
# (Total Produksi EBT per Tahun & Rata-Rata Produksi Bulanan per Jenis PLT)
# ============================================================

# Total produksi EBT per tahun (2026, 2027, 2028)
ringkasan_total_per_tahun = forecast_total.groupby("Tahun", as_index=False)["Total_Produksi_EBT"] \
                                           .sum().rename(columns={"Total_Produksi_EBT": "Total_Produksi_EBT_Tahunan"})

print("Ringkasan Total Produksi EBT per Tahun (Hasil Forecasting):")
display(ringkasan_total_per_tahun)

# Rata-rata produksi bulanan untuk setiap jenis PLT selama periode forecasting
ringkasan_rata2_bulanan_per_plt = forecast_ebt.groupby("Jenis_PLT", as_index=False)["Produksi_Prediksi"] \
                                               .mean().rename(columns={"Produksi_Prediksi": "Rata2_Produksi_Bulanan"})
ringkasan_rata2_bulanan_per_plt = ringkasan_rata2_bulanan_per_plt.sort_values(
    by="Rata2_Produksi_Bulanan", ascending=False
).reset_index(drop=True)

print("\nRingkasan Rata-Rata Produksi Bulanan per Jenis PLT (2026-2028):")
display(ringkasan_rata2_bulanan_per_plt)

In [ ]:
# ============================================================
# KEGIATAN 1: Import Library Tambahan dan Membuat Struktur Folder
# (shutil untuk copy file & zip, google.colab.files untuk download)
# ============================================================

import os
import shutil
from google.colab import files

# Folder utama hasil penelitian
ROOT_FOLDER = "EBT_LSTM_Streamlit"

# Sub-folder sesuai struktur yang diminta
SUBFOLDERS = ["models", "scalers", "config", "evaluation", "forecast"]

# Menghapus folder lama jika sudah ada (agar tidak duplikat saat cell dijalankan ulang)
if os.path.exists(ROOT_FOLDER):
    shutil.rmtree(ROOT_FOLDER)

# Membuat folder utama beserta seluruh sub-foldernya
for sub in SUBFOLDERS:
    os.makedirs(os.path.join(ROOT_FOLDER, sub), exist_ok=True)

print(f"Struktur folder '{ROOT_FOLDER}/' berhasil dibuat dengan sub-folder: {SUBFOLDERS}")

In [ ]:
# ============================================================
# KEGIATAN 2: Menyalin Seluruh Model Terbaik (.keras) ke models/
# (Model diambil dari model_final_per_plt, TIDAK dilatih ulang)
# ============================================================

for plt_name, info in model_final_per_plt.items():
    nama_file_aman = plt_name.replace(" ", "_")
    path_sumber = os.path.join(FOLDER_MODEL_FINAL, f"model_final_{nama_file_aman}.keras")
    path_tujuan = os.path.join(ROOT_FOLDER, "models", f"model_final_{nama_file_aman}.keras")

    shutil.copy(path_sumber, path_tujuan)
    print(f"Model {plt_name:12s} ({info['metode']:18s}) disalin ke: {path_tujuan}")

In [ ]:
# ============================================================
# KEGIATAN 2: Menyalin Seluruh Model Terbaik (.keras) ke models/
# (Model diambil dari model_final_per_plt, TIDAK dilatih ulang)
# ============================================================

for plt_name, info in model_final_per_plt.items():
    nama_file_aman = plt_name.replace(" ", "_")
    path_sumber = os.path.join(FOLDER_MODEL_FINAL, f"model_final_{nama_file_aman}.keras")
    path_tujuan = os.path.join(ROOT_FOLDER, "models", f"model_final_{nama_file_aman}.keras")

    shutil.copy(path_sumber, path_tujuan)
    print(f"Model {plt_name:12s} ({info['metode']:18s}) disalin ke: {path_tujuan}")

In [ ]:
# ============================================================
# KEGIATAN 3: Menyalin Seluruh Scaler (.pkl) ke scalers/
# ============================================================

for plt_name in metadata_model.keys():
    nama_file_aman = plt_name.replace(" ", "_")

    for jenis_scaler in ["scaler_fitur", "scaler_target"]:
        path_sumber = os.path.join(FOLDER_MODEL_FINAL, f"{jenis_scaler}_{nama_file_aman}.pkl")
        path_tujuan = os.path.join(ROOT_FOLDER, "scalers", f"{jenis_scaler}_{nama_file_aman}.pkl")
        shutil.copy(path_sumber, path_tujuan)

    print(f"Scaler fitur & target untuk {plt_name} disalin ke folder scalers/")

In [ ]:
# ============================================================
# KEGIATAN 4: Menyimpan Konfigurasi Model (per PLT) ke config/
# Berisi: nama PLT, metode, window size, learning rate, dropout,
# batch size, optimizer, epoch, fitur input, dan target
# ============================================================

konfigurasi_final_per_plt = {}

for plt_name, meta in metadata_model.items():
    konfigurasi_final_per_plt[plt_name] = {
        "jenis_plt": plt_name,
        "metode": meta["metode_pelatihan"],           # Baseline/Direct Training atau Transfer Learning
        "window_size": meta["window_size"],
        "learning_rate": meta["learning_rate"],
        "dropout": meta["dropout"],
        "batch_size": meta["batch_size"],
        "epochs": meta["epochs"],
        "optimizer": meta["optimizer"],
        "fitur_input": meta["fitur_input"],
        "target_prediksi": meta["target_prediksi"],
        "nama_file_model": f"model_final_{plt_name.replace(' ', '_')}.keras",
        "nama_file_scaler_fitur": f"scaler_fitur_{plt_name.replace(' ', '_')}.pkl",
        "nama_file_scaler_target": f"scaler_target_{plt_name.replace(' ', '_')}.pkl",
    }

path_config = os.path.join(ROOT_FOLDER, "config", "konfigurasi_model.json")
with open(path_config, "w", encoding="utf-8") as f:
    json.dump(konfigurasi_final_per_plt, f, indent=4, ensure_ascii=False)

print(f"Konfigurasi model seluruh PLT disimpan ke: {path_config}")

# ============================================================
# [TASK GROUP 3] Menyusun evaluasi_final.csv
#
# Model Final dilatih memakai SELURUH data 2023-2025 TANPA menyisakan data
# uji (lihat catatan di awal file) -- jadi baris 2025 sudah pernah dilihat
# model ini saat training, dan mengevaluasinya langsung pada 2025 akan
# menyesatkan (bukan metrik generalisasi yang jujur, hasilnya pasti
# terlihat bagus karena in-sample).
#
# Sebagai gantinya, evaluasi_final.csv memakai RMSE/MAE/MAPE test 2025 yang
# SUDAH dihitung di Task Group 2 (RMSE_Test_2025_Konfirmasi di
# model_terbaik_per_plt) -- itu angka yang genuinely held-out, dari model
# dengan arsitektur & konfigurasi yang SAMA persis dengan Model Final
# (sebelum dilatih ulang dengan seluruh data). Ini estimasi performa yang
# dibawa dari tahap seleksi, BUKAN pengukuran langsung terhadap
# model_final_per_plt.
# ============================================================

peta_kolom_mae_test = {
    "Baseline": "MAE_Baseline", "FineTuning": "MAE_FineTuning",
    "Iterasi1": "MAE_Iterasi1", "Iterasi2": "MAE_Iterasi2", "Iterasi3": "MAE_Iterasi3",
}
peta_kolom_mape_test = {
    "Baseline": "MAPE_Baseline", "FineTuning": "MAPE_FineTuning",
    "Iterasi1": "MAPE_Iterasi1", "Iterasi2": "MAPE_Iterasi2", "Iterasi3": "MAPE_Iterasi3",
}

evaluasi_final = model_terbaik_per_plt[
    ["Jenis_PLT", "Model_Terbaik", "RMSE_Test_2025_Konfirmasi"]
].copy()
evaluasi_final["MAE_Test_2025_Konfirmasi"] = evaluasi_final.apply(
    lambda baris: perbandingan_model_indexed.loc[baris["Jenis_PLT"], peta_kolom_mae_test[baris["Model_Terbaik"]]],
    axis=1,
)
evaluasi_final["MAPE_Test_2025_Konfirmasi"] = evaluasi_final.apply(
    lambda baris: perbandingan_model_indexed.loc[baris["Jenis_PLT"], peta_kolom_mape_test[baris["Model_Terbaik"]]],
    axis=1,
)
evaluasi_final = evaluasi_final.rename(columns={"Model_Terbaik": "Metode_Terpilih"})
evaluasi_final = evaluasi_final.sort_values(by="RMSE_Test_2025_Konfirmasi", ascending=True).reset_index(drop=True)

print("Estimasi performa Model Final per PLT (dibawa dari konfirmasi test 2025 Task Group 2):")
evaluasi_final

In [ ]:
# ============================================================
# KEGIATAN 5: Menyimpan Seluruh Hasil Evaluasi (CSV) ke evaluation/
# (Baseline, Fine-Tuning, Iterasi 1-3, Perbandingan Lengkap, Model Terbaik)
# ============================================================

daftar_evaluasi_disimpan = {
    "evaluasi_final.csv": evaluasi_final,
    "evaluasi_baseline.csv": df_evaluasi,
    "evaluasi_finetuning.csv": df_evaluasi_finetune,
    "evaluasi_iterasi1.csv": hasil_iterasi1,
    "evaluasi_iterasi2.csv": hasil_iterasi2,
    "evaluasi_iterasi3.csv": hasil_iterasi3,
    "perbandingan_lengkap_seluruh_model.csv": df_perbandingan_lengkap,
    # [TASK GROUP 2] ringkasan_rata2_seluruh_model.csv sekarang berisi
    # ringkasan_model_validasi (dasar pemilihan model_terbaik), bukan lagi
    # ringkasan_model (test 2025) -- ringkasan_model test 2025 tetap ada di
    # perbandingan_lengkap_seluruh_model.csv untuk pelaporan/konfirmasi.
    "ringkasan_rata2_seluruh_model.csv": ringkasan_model_validasi,
    "model_terbaik_per_plt.csv": model_terbaik_per_plt,
    "perbandingan_model_VALIDASI.csv": perbandingan_model_VALIDASI,
}

for nama_file, df in daftar_evaluasi_disimpan.items():
    path_tujuan = os.path.join(ROOT_FOLDER, "evaluation", nama_file)
    df.to_csv(path_tujuan, index=False)
    print(f"Disimpan: {path_tujuan}")

In [ ]:
# ============================================================
# KEGIATAN 6: Menyimpan Hasil Forecasting 2026-2028 (CSV) ke forecast/
# ============================================================

path_forecast_per_plt = os.path.join(ROOT_FOLDER, "forecast", "forecast_per_PLT_2026_2028.csv")
path_forecast_total = os.path.join(ROOT_FOLDER, "forecast", "forecast_total_2026_2028.csv")

forecast_ebt.to_csv(path_forecast_per_plt, index=False)
forecast_total.to_csv(path_forecast_total, index=False)

print(f"Disimpan: {path_forecast_per_plt}")
print(f"Disimpan: {path_forecast_total}")

In [ ]:
# ============================================================
# KEGIATAN 7: Menyimpan Metadata Penelitian ke Root Folder
# (Ringkasan umum penelitian untuk keperluan dokumentasi & Streamlit)
# ============================================================

metadata_penelitian = {
    "judul_penelitian": "Prediksi Produksi Energi Baru Terbarukan (EBT) Sektor Kelistrikan "
                         "Provinsi Sulawesi Selatan Menggunakan LSTM dengan Transfer Learning",
    "metodologi": "CRISP-DM",
    "jenis_plt": list(metadata_model.keys()),
    "periode_data_historis": "2023-2025 (Regional), 2020-2024 (Nasional untuk Pre-Training)",
    "periode_forecasting": "Januari 2026 - Desember 2028",
    # [TASK GROUP 3] "model_terbaik_keseluruhan"/"konfigurasi_model_terbaik_keseluruhan"
    # (satu model/konfigurasi global) DIHAPUS -- sudah tidak representatif
    # karena tiap jenis PLT sekarang punya konfigurasi final sendiri
    # (lihat "konfigurasi_final_per_plt" dan "metode_per_plt" di bawah).
    "konfigurasi_final_per_plt": konfigurasi_final_per_plt,
    "metode_per_plt": {plt: info["metode"] for plt, info in model_final_per_plt.items()},
    "tanggal_dibuat": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "catatan": "Forecasting menggunakan pendekatan autoregressive dengan asumsi fitur "
               "Cuaca dan Kapasitas mengikuti Last Observation Carried Forward (LOCF). "
               "Konfigurasi model final (window size, learning rate, dropout, batch "
               "size, epochs, metode) ditentukan PER JENIS PLT berdasarkan RMSE "
               "validasi masing-masing (lihat konfigurasi_final_per_plt), bukan satu "
               "konfigurasi tunggal untuk seluruh PLT."
}

path_metadata_penelitian = os.path.join(ROOT_FOLDER, "metadata_penelitian.json")
with open(path_metadata_penelitian, "w", encoding="utf-8") as f:
    json.dump(metadata_penelitian, f, indent=4, ensure_ascii=False)

print(f"Metadata penelitian disimpan ke: {path_metadata_penelitian}")

In [ ]:
# ============================================================
# KEGIATAN 8: Menampilkan Struktur Folder yang Telah Dibuat
# (Verifikasi sebelum dikompres menjadi ZIP)
# ============================================================

print(f"Struktur folder '{ROOT_FOLDER}/':\n")

for root, dirs, filenames in os.walk(ROOT_FOLDER):
    level = root.replace(ROOT_FOLDER, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for filename in filenames:
        print(f"{indent}  - {filename}")

In [ ]:
# ============================================================
# KEGIATAN 9: Mengompres Folder EBT_LSTM_Streamlit/ Menjadi File ZIP
# ============================================================

NAMA_ZIP = "EBT_LSTM_Streamlit"  # tanpa ekstensi .zip (otomatis ditambahkan oleh shutil)

# shutil.make_archive menghasilkan file "EBT_LSTM_Streamlit.zip" dari folder ROOT_FOLDER
path_zip = shutil.make_archive(base_name=NAMA_ZIP, format="zip", root_dir=ROOT_FOLDER)

# Menampilkan ukuran file ZIP (dalam MB) untuk verifikasi
ukuran_mb = os.path.getsize(path_zip) / (1024 * 1024)
print(f"File ZIP berhasil dibuat: {path_zip}")
print(f"Ukuran file: {ukuran_mb:.2f} MB")

In [ ]:
# ============================================================
# KEGIATAN 10: Mengunduh File ZIP dari Google Colab
# ============================================================

print("Memulai proses download file ZIP...")
files.download(path_zip)
print("Jika unduhan tidak dimulai otomatis, periksa pop-up blocker pada browser Anda.")